## Product Recommendation Modeling & Lead Generation

This notebook builds product recommendations, lead scores, expected order values and probability-weighted revenue opportunities for B2B customers.
The business idea is:
1. Use customer behavior, cohort, value, activity, and Salesforce features.
2. Train product-level models that estimate how likely a customer is to buy each product group.
3. Exclude products the customer has already purchased.
4. Rank the remaining products as recommendations.
5. Combine model probability with business value and customer activity into a lead score.

> **Interpretation warning:** the current target is an all-history `purchase_flag`, not a future purchase within a fixed horizon. Therefore `model_probability × expected_order_value` is currently a **propensity-weighted revenue opportunity proxy**, not a finance-grade revenue forecast. It becomes literal expected revenue only after using a time-based future target and calibrated probabilities.

In [2]:
#Setup
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROCESSED_DIR = Path("../data/processed")
FEATURE_DIR = Path("../data/feature_engineering")
CLUSTER_DIR = Path("../data/cohort_clustering")
OUTPUT_DIR = Path("../data/product_recommendation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TOP_K = 3
MIN_POSITIVES = 25

# expected-order-value assumptions
VALUE_SHRINKAGE_STRENGTH = 30
CUSTOMER_VALUE_MULTIPLIER_BOUNDS = (0.50, 2.00)
WINSOR_LIMITS = (0.01, 0.99)

# false for the current all-history affinity model. Change only after rebuilding
# the target as a future purchase within a fixed horizon and calibrating probabilities.
MODEL_PROBABILITY_IS_CALIBRATED_FUTURE_PROBABILITY = False

print("Processed-data directory:", PROCESSED_DIR.resolve())
print("Feature directory:", FEATURE_DIR.resolve())
print("Cluster directory:", CLUSTER_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())


Processed-data directory: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\lead-generation\data\processed
Feature directory: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\lead-generation\data\feature_engineering
Cluster directory: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\lead-generation\data\cohort_clustering
Output directory: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\lead-generation\data\product_recommendation


In [3]:
# Load modeling inputs
df_sales_full = pd.read_csv(PROCESSED_DIR / "df_sales_full.csv")
df_customers = pd.read_csv(CLUSTER_DIR / "df_clustering_output.csv")
cust_prod_matrix_grp_1 = pd.read_csv(FEATURE_DIR / "cust_prod_matrix_grp_1.csv")
cust_prod_matrix_grp_3 = pd.read_csv(FEATURE_DIR / "cust_prod_matrix_grp_3.csv")

print("Historical sales table:", df_sales_full.shape)
print("Customer feature table:", df_customers.shape)
print("Product group 1 matrix:", cust_prod_matrix_grp_1.shape)
print("Product group 3 matrix:", cust_prod_matrix_grp_3.shape)

display(df_sales_full.head())
display(df_customers.head())
display(cust_prod_matrix_grp_1.head())
display(cust_prod_matrix_grp_3.head())

Historical sales table: (68723, 24)
Customer feature table: (1993, 85)
Product group 1 matrix: (1993, 7)
Product group 3 matrix: (1993, 36)


,Date,ID_Customer,ID_Product,Sales,Units,YearMonth,Year,Month,Customer_Name,Customer_Segment,Customer_Size,Industry,Region_CC,Region_OC,Country,Acquisition_Channel,Customer_Start_Date,Product_Name,Prod_Line,Prd_Grp_1,Prd_Grp_2,Prd_Grp_3,Unit_Price,Margin_Category
0,2023-02-01,C00001,P0116,"2,301.4400",13,2023-02,2023,2,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01,Premium Standard Type 2 0116,Consumables,Premium Products,Premium Standard,Premium Standard Type 2,194.3500,Low
1,2023-02-01,C00001,P0083,"1,545.1500",6,2023-02,2023,2,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01,Premium Plus Type 1 0083,Maintenance,Premium Products,Premium Plus,Premium Plus Type 1,285.9000,High
2,2023-04-01,C00001,P0160,763.5000,15,2023-04,2023,4,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01,Repair Kits Type 1 0160,Industrial Equipment,Maintenance Kits,Repair Kits,Repair Kits Type 1,54.5600,Low
3,2023-05-01,C00001,P0170,805.0300,19,2023-05,2023,5,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01,Core Basic Type 2 0170,Industrial Equipment,Core Products,Core Basic,Core Basic Type 2,42.5400,Medium
4,2023-05-01,C00001,P0160,155.1000,3,2023-05,2023,5,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01,Repair Kits Type 1 0160,Industrial Equipment,Maintenance Kits,Repair Kits,Repair Kits Type 1,54.5600,Low


,ID_Customer,Sales,Units,order_lines,active_purchase_months,distinct_products,prd_line_coverage_pct,prd_grp_1_coverage_pct,prd_grp_2_coverage_pct,prd_grp_3_coverage_pct,avg_order_line_value,avg_unit_price_realized,active_months,lifetime_months,months_since_last_purchase,activity_ratio_lifetime,max_consec_active_months,current_consec_active_months,avg_inactive_gap_months,max_inactive_gap_months,is_inactive_6m,is_inactive_12m,avg_monthly_revenue,historical_ltv,purchase_freq_per_year,R_recency_m,F_frequency_m,M_sales,sf_active_months,sf_active_in_last_3m,sf_active_in_last_6m,sf_active_in_last_12m,sf_total_events,sf_total_selling_time,sf_total_activity_time,sf_activity_ratio_lifetime,sf_events_per_active_month,has_salesforce_activity,sales_without_recent_sf_activity,sf_activity_without_recent_sales,sales_value_segment,recency_segment,purchase_share_grp1_Automation Modules,purchase_share_grp1_Core Products,purchase_share_grp1_Maintenance Kits,purchase_share_grp1_Premium Products,purchase_share_grp1_Safety Solutions,purchase_share_grp1_Service Contracts,purchase_share_grp3_Calibration Kits Type 1,purchase_share_grp3_Calibration Kits Type 2,purchase_share_grp3_Compliance Safety Type 1,purchase_share_grp3_Compliance Safety Type 2,purchase_share_grp3_Controllers Type 1,purchase_share_grp3_Controllers Type 2,purchase_share_grp3_Core Advanced Type 1,purchase_share_grp3_Core Advanced Type 2,purchase_share_grp3_Core Basic Type 1,purchase_share_grp3_Core Basic Type 2,purchase_share_grp3_Core Replacement Type 1,purchase_share_grp3_Core Replacement Type 2,purchase_share_grp3_Extended Support Type 1,purchase_share_grp3_Extended Support Type 2,purchase_share_grp3_Facility Safety Type 2,purchase_share_grp3_Installation Type 1,purchase_share_grp3_Installation Type 2,purchase_share_grp3_Monitoring Type 1,purchase_share_grp3_Monitoring Type 2,purchase_share_grp3_Personal Safety Type 1,purchase_share_grp3_Personal Safety Type 2,purchase_share_grp3_Premium Custom Type 1,purchase_share_grp3_Premium Custom Type 2,purchase_share_grp3_Premium Plus Type 1,purchase_share_grp3_Premium Plus Type 2,purchase_share_grp3_Premium Standard Type 1,purchase_share_grp3_Premium Standard Type 2,purchase_share_grp3_Preventive Kits Type 1,purchase_share_grp3_Preventive Kits Type 2,purchase_share_grp3_Repair Kits Type 1,purchase_share_grp3_Repair Kits Type 2,purchase_share_grp3_Sensors Type 1,purchase_share_grp3_Sensors Type 2,purchase_share_grp3_Training Type 1,purchase_share_grp3_Training Type 2,cluster,cohort_ML
0,C00001,"144,433.2700",653,47,23,5,0.5714,0.6667,0.2778,0.1429,"3,073.0483",221.1842,23,34,1,0.6765,7.0000,3.0000,0.5000,3.0000,0,0,"4,248.0374","144,433.2700",8.1176,1,23,"144,433.2700",29,1,1,1,117,81.9200,67.5900,0.8056,4.0345,1,0,0,Top Value,Very Recent,0.1914,0.1654,0.1654,0.4778,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1654,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2665,0.0000,0.0000,0.2113,0.0000,0.0000,0.1654,0.0000,0.1914,0.0000,0.0000,0.0000,0,Emerging
1,C00002,"4,738.6100",21,17,8,9,1.0000,0.6667,0.3333,0.2286,278.7418,225.6481,8,31,1,0.2581,3.0000,1.0000,3.2857,14.0000,0,0,152.8584,"4,738.6100",3.0968,1,8,"4,738.6100",13,1,1,1,45,28.8700,18.8400,0.4333,3.4615,1,0,0,Low Value,Very Recent,0.3333,0.3333,0.0000,0.1905,0.1429,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1905,0.0000,0.1429,0.0952,0.0952,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1429,0.0000,0.0000,0.0000,0.0000,0.0000,0.0952,0.0952,0.0000,0.0000,0.0000,0.0000,0.1429,0.0000,0.0000,0.0000,0,Emerging
2,C00003,"199,361.8000",658,105,31,11,0.5714,0.8333,0.5000,0.2857,"1,898.6838",302.9815,31,35,1,0.8857,15.0000,4.0000,0.1333,1.0000,0,0,"5,696.0514","199,361.8000",10.6286,1,31,"199,361.8000",24,1,1,1,124,76.3000,70.7800,0.7273,5.1667,1,0,0,Top Value,Very Recent,0.0000,0.3283,0.1201,0.1505,0.1474,0.2538,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0881,0.2401,0.0000,0.0000,0.0000,

,ID_Customer,purchase_flag_grp1_Automation Modules,purchase_flag_grp1_Core Products,purchase_flag_grp1_Maintenance Kits,purchase_flag_grp1_Premium Products,purchase_flag_grp1_Safety Solutions,purchase_flag_grp1_Service Contracts
0,C00001,1,1,1,1,0,0
1,C00002,1,1,0,1,1,0
2,C00003,0,1,1,1,1,1
3,C00004,1,1,1,0,0,0
4,C00005,1,1,0,1,0,1


,ID_Customer,purchase_flag_grp3_Calibration Kits Type 1,purchase_flag_grp3_Calibration Kits Type 2,purchase_flag_grp3_Compliance Safety Type 1,purchase_flag_grp3_Compliance Safety Type 2,purchase_flag_grp3_Controllers Type 1,purchase_flag_grp3_Controllers Type 2,purchase_flag_grp3_Core Advanced Type 1,purchase_flag_grp3_Core Advanced Type 2,purchase_flag_grp3_Core Basic Type 1,purchase_flag_grp3_Core Basic Type 2,purchase_flag_grp3_Core Replacement Type 1,purchase_flag_grp3_Core Replacement Type 2,purchase_flag_grp3_Extended Support Type 1,purchase_flag_grp3_Extended Support Type 2,purchase_flag_grp3_Facility Safety Type 2,purchase_flag_grp3_Installation Type 1,purchase_flag_grp3_Installation Type 2,purchase_flag_grp3_Monitoring Type 1,purchase_flag_grp3_Monitoring Type 2,purchase_flag_grp3_Personal Safety Type 1,purchase_flag_grp3_Personal Safety Type 2,purchase_flag_grp3_Premium Custom Type 1,purchase_flag_grp3_Premium Custom Type 2,purchase_flag_grp3_Premium Plus Type 1,purchase_flag_grp3_Premium Plus Type 2,purchase_flag_grp3_Premium Standard Type 1,purchase_flag_grp3_Premium Standard Type 2,purchase_flag_grp3_Preventive Kits Type 1,purchase_flag_grp3_Preventive Kits Type 2,purchase_flag_grp3_Repair Kits Type 1,purchase_flag_grp3_Repair Kits Type 2,purchase_flag_grp3_Sensors Type 1,purchase_flag_grp3_Sensors Type 2,purchase_flag_grp3_Training Type 1,purchase_flag_grp3_Training Type 2
0,C00001,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,1,0,0,0
1,C00002,0,0,0,0,0,1,0,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0
2,C00003,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,1,1,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,1,0
3,C00004,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
4,C00005,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0


## Inspect Recommendation Targets

In [4]:
# understand imbalances
def target_summary(matrix: pd.DataFrame, prefix: str) -> pd.DataFrame:
    target_cols = [c for c in matrix.columns if c.startswith(prefix)]
    rows = []
    for col in target_cols:
        y = matrix[col].astype(int)
        rows.append({
            "target_col": col,
            "product": col.replace(prefix, ""),
            "customers": len(y),
            "positive_customers": int(y.sum()),
            "positive_rate": float(y.mean()),
            "negative_customers": int((1 - y).sum())})
    return pd.DataFrame(rows).sort_values("positive_rate", ascending=False)


grp1_target_summary = target_summary(cust_prod_matrix_grp_1, "purchase_flag_grp1_")
grp3_target_summary = target_summary(cust_prod_matrix_grp_3, "purchase_flag_grp3_")

display(grp1_target_summary)
display(grp3_target_summary.head(20))

,target_col,product,customers,positive_customers,positive_rate,negative_customers
0,purchase_flag_grp1_Automation Modules,Automation Modules,1993,1656,0.8309,337
1,purchase_flag_grp1_Core Products,Core Products,1993,1622,0.8138,371
2,purchase_flag_grp1_Maintenance Kits,Maintenance Kits,1993,1310,0.6573,683
5,purchase_flag_grp1_Service Contracts,Service Contracts,1993,1174,0.5891,819
3,purchase_flag_grp1_Premium Products,Premium Products,1993,1055,0.5294,938
4,purchase_flag_grp1_Safety Solutions,Safety Solutions,1993,909,0.4561,1084


,target_col,product,customers,positive_customers,positive_rate,negative_customers
7,purchase_flag_grp3_Core Advanced Type 2,Core Advanced Type 2,1993,695,0.3487,1298
4,purchase_flag_grp3_Controllers Type 1,Controllers Type 1,1993,684,0.3432,1309
9,purchase_flag_grp3_Core Basic Type 2,Core Basic Type 2,1993,678,0.3402,1315
31,purchase_flag_grp3_Sensors Type 1,Sensors Type 1,1993,669,0.3357,1324
6,purchase_flag_grp3_Core Advanced Type 1,Core Advanced Type 1,1993,609,0.3056,1384
8,purchase_flag_grp3_Core Basic Type 1,Core Basic Type 1,1993,601,0.3016,1392
32,purchase_flag_grp3_Sensors Type 2,Sensors Type 2,1993,586,0.2940,1407
5,purchase_flag_grp3_Controllers Type 2,Controllers Type 2,1993,585,0.2935,1408
17,purchase_flag_grp3_Monitoring Type 1,Monitoring Type 1,1993,459,0.2303,1534
30,purchase_flag_grp3_Repair Kits Type 2,Repair Kits Type 2,1993,428,0.2148,1565


## Build Modeling Tables

In [5]:
# merge customer features with product target matrix

def build_modeling_table(customer_df: pd.DataFrame, target_matrix: pd.DataFrame) -> pd.DataFrame:
    # validate one row per customer before merging
    if customer_df["ID_Customer"].duplicated().any():
        raise ValueError("customer_df contains duplicated ID_Customer values.")
    if target_matrix["ID_Customer"].duplicated().any():
        raise ValueError("target_matrix contains duplicated ID_Customer values.")

    return customer_df.merge(target_matrix, on="ID_Customer", how="inner", validate="one_to_one")


df_model_grp1 = build_modeling_table(df_customers, cust_prod_matrix_grp_1)
df_model_grp3 = build_modeling_table(df_customers, cust_prod_matrix_grp_3)

print("Modeling table group 1:", df_model_grp1.shape)
print("Modeling table group 3:", df_model_grp3.shape)

display(df_model_grp1.head())

Modeling table group 1: (1993, 91)
Modeling table group 3: (1993, 120)


,ID_Customer,Sales,Units,order_lines,active_purchase_months,distinct_products,prd_line_coverage_pct,prd_grp_1_coverage_pct,prd_grp_2_coverage_pct,prd_grp_3_coverage_pct,avg_order_line_value,avg_unit_price_realized,active_months,lifetime_months,months_since_last_purchase,activity_ratio_lifetime,max_consec_active_months,current_consec_active_months,avg_inactive_gap_months,max_inactive_gap_months,is_inactive_6m,is_inactive_12m,avg_monthly_revenue,historical_ltv,purchase_freq_per_year,R_recency_m,F_frequency_m,M_sales,sf_active_months,sf_active_in_last_3m,sf_active_in_last_6m,sf_active_in_last_12m,sf_total_events,sf_total_selling_time,sf_total_activity_time,sf_activity_ratio_lifetime,sf_events_per_active_month,has_salesforce_activity,sales_without_recent_sf_activity,sf_activity_without_recent_sales,sales_value_segment,recency_segment,purchase_share_grp1_Automation Modules,purchase_share_grp1_Core Products,purchase_share_grp1_Maintenance Kits,purchase_share_grp1_Premium Products,purchase_share_grp1_Safety Solutions,purchase_share_grp1_Service Contracts,purchase_share_grp3_Calibration Kits Type 1,purchase_share_grp3_Calibration Kits Type 2,purchase_share_grp3_Compliance Safety Type 1,purchase_share_grp3_Compliance Safety Type 2,purchase_share_grp3_Controllers Type 1,purchase_share_grp3_Controllers Type 2,purchase_share_grp3_Core Advanced Type 1,purchase_share_grp3_Core Advanced Type 2,purchase_share_grp3_Core Basic Type 1,purchase_share_grp3_Core Basic Type 2,purchase_share_grp3_Core Replacement Type 1,purchase_share_grp3_Core Replacement Type 2,purchase_share_grp3_Extended Support Type 1,purchase_share_grp3_Extended Support Type 2,purchase_share_grp3_Facility Safety Type 2,purchase_share_grp3_Installation Type 1,purchase_share_grp3_Installation Type 2,purchase_share_grp3_Monitoring Type 1,purchase_share_grp3_Monitoring Type 2,purchase_share_grp3_Personal Safety Type 1,purchase_share_grp3_Personal Safety Type 2,purchase_share_grp3_Premium Custom Type 1,purchase_share_grp3_Premium Custom Type 2,purchase_share_grp3_Premium Plus Type 1,purchase_share_grp3_Premium Plus Type 2,purchase_share_grp3_Premium Standard Type 1,purchase_share_grp3_Premium Standard Type 2,purchase_share_grp3_Preventive Kits Type 1,purchase_share_grp3_Preventive Kits Type 2,purchase_share_grp3_Repair Kits Type 1,purchase_share_grp3_Repair Kits Type 2,purchase_share_grp3_Sensors Type 1,purchase_share_grp3_Sensors Type 2,purchase_share_grp3_Training Type 1,purchase_share_grp3_Training Type 2,cluster,cohort_ML,purchase_flag_grp1_Automation Modules,purchase_flag_grp1_Core Products,purchase_flag_grp1_Maintenance Kits,purchase_flag_grp1_Premium Products,purchase_flag_grp1_Safety Solutions,purchase_flag_grp1_Service Contracts
0,C00001,"144,433.2700",653,47,23,5,0.5714,0.6667,0.2778,0.1429,"3,073.0483",221.1842,23,34,1,0.6765,7.0000,3.0000,0.5000,3.0000,0,0,"4,248.0374","144,433.2700",8.1176,1,23,"144,433.2700",29,1,1,1,117,81.9200,67.5900,0.8056,4.0345,1,0,0,Top Value,Very Recent,0.1914,0.1654,0.1654,0.4778,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1654,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2665,0.0000,0.0000,0.2113,0.0000,0.0000,0.1654,0.0000,0.1914,0.0000,0.0000,0.0000,0,Emerging,1,1,1,1,0,0
1,C00002,"4,738.6100",21,17,8,9,1.0000,0.6667,0.3333,0.2286,278.7418,225.6481,8,31,1,0.2581,3.0000,1.0000,3.2857,14.0000,0,0,152.8584,"4,738.6100",3.0968,1,8,"4,738.6100",13,1,1,1,45,28.8700,18.8400,0.4333,3.4615,1,0,0,Low Value,Very Recent,0.3333,0.3333,0.0000,0.1905,0.1429,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1905,0.0000,0.1429,0.0952,0.0952,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1429,0.0000,0.0000,0.0000,0.0000,0.0000,0.0952,0.0952,0.0000,0.0000,0.0000,0.0000,0.1429,0.0000,0.0000,0.0000,0,Emerging,1,1,0,1,1,0
2,C00003,"199,361.8000",658,105,31,11,0.5714,0.8333,0.5000,0.2857,"1,898.6838",302.9815,31,35,1,0.8857,15.0000,4.0000,0.1333,1.0000,0,0,"5,696.0514",

## Define Feature Exclusions

In [6]:
def get_target_cols(df: pd.DataFrame, target_prefix: str) -> list[str]:
    return [c for c in df.columns if c.startswith(target_prefix)]

def base_columns_to_drop(df: pd.DataFrame) -> list[str]:
    drop_cols = []

    # remove ID, date, period columns
    drop_cols += [c for c in df.columns if c.lower() in ["id_customer", "id_product"]]
    drop_cols += [c for c in df.columns if "date" in c.lower()]
    drop_cols += [c for c in df.columns if c.lower().endswith("_ym")]
    drop_cols += [c for c in df.columns if c.lower().endswith("_ym_ord")]

    # Drop all purchase-share columns to prevent target leakage
    drop_cols += [c for c in df.columns if c.startswith("purchase_share_")]

    # remove duplicate column names while preserving order
    return list(dict.fromkeys([c for c in drop_cols if c in df.columns]))


grp1_target_cols = get_target_cols(df_model_grp1, "purchase_flag_grp1_")
grp3_target_cols = get_target_cols(df_model_grp3, "purchase_flag_grp3_")

print("Group 1 targets:", len(grp1_target_cols))
print("Group 3 targets:", len(grp3_target_cols))
print("Base drop columns example:", base_columns_to_drop(df_model_grp1)[:20])


Group 1 targets: 6
Group 3 targets: 35
Base drop columns example: ['ID_Customer', 'purchase_share_grp1_Automation Modules', 'purchase_share_grp1_Core Products', 'purchase_share_grp1_Maintenance Kits', 'purchase_share_grp1_Premium Products', 'purchase_share_grp1_Safety Solutions', 'purchase_share_grp1_Service Contracts', 'purchase_share_grp3_Calibration Kits Type 1', 'purchase_share_grp3_Calibration Kits Type 2', 'purchase_share_grp3_Compliance Safety Type 1', 'purchase_share_grp3_Compliance Safety Type 2', 'purchase_share_grp3_Controllers Type 1', 'purchase_share_grp3_Controllers Type 2', 'purchase_share_grp3_Core Advanced Type 1', 'purchase_share_grp3_Core Advanced Type 2', 'purchase_share_grp3_Core Basic Type 1', 'purchase_share_grp3_Core Basic Type 2', 'purchase_share_grp3_Core Replacement Type 1', 'purchase_share_grp3_Core Replacement Type 2', 'purchase_share_grp3_Extended Support Type 1']


## Preprocessing Helper 

In [7]:
def make_preprocessor(X: pd.DataFrame) -> tuple[ColumnTransformer, list[str], list[str]]:
    numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())])

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_cols),
            ("cat", categorical_pipeline, categorical_cols)],
        remainder="drop")

    return preprocessor, numeric_cols, categorical_cols

## Baseline Model: Popularity Ranking

In [8]:
def popularity_baseline_score(y_valid: pd.Series, train_positive_rate: float) -> dict:
    # every customer receives the same probability: the product's training positive rate
    baseline_proba = np.repeat(train_positive_rate, len(y_valid))

    try:
        baseline_ap = average_precision_score(y_valid, baseline_proba) 
    except ValueError:
        baseline_ap = np.nan

    try:
        baseline_auc = roc_auc_score(y_valid, baseline_proba)
    except ValueError:
        baseline_auc = np.nan

    return {
        "baseline_avg_precision": baseline_ap,
        "baseline_roc_auc": baseline_auc}

## Train and Evaluate One Product Model

In [9]:
def train_one_product_model(
    df: pd.DataFrame,
    target_col: str,
    all_target_cols: list[str],
    product_prefix: str,
) -> tuple[Pipeline | None, dict]:
    y = df[target_col].astype(int)
    product_name = target_col.replace(product_prefix, "")

    # Skip products with too little signal
    if y.nunique() < 2 or y.sum() < MIN_POSITIVES:
        return None, {
            "product": product_name,
            "target_col": target_col,
            "status": "skipped_low_support",
            "n_total": len(y),
            "positive_customers": int(y.sum()),
            "positive_rate": float(y.mean())}
    
    # drop target columns and other non suitable columns
    leakage_cols = all_target_cols
    drop_cols = base_columns_to_drop(df) + leakage_cols
    X = df.drop(columns=drop_cols, errors="ignore").copy()

    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y)

    baseline = popularity_baseline_score(y_valid, y_train.mean())
    preprocessor, numeric_cols, categorical_cols = make_preprocessor(X_train)

    candidates = {
        "LogisticRegression": LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE),
        "RandomForest": RandomForestClassifier(
            n_estimators=350,
            min_samples_leaf=8,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1)}

    fitted_models = []

    for model_name, model in candidates.items():
        pipe = Pipeline(
            steps=[
                ("preprocess", preprocessor),
                ("model", model)])

        pipe.fit(X_train, y_train)
        proba_valid = pipe.predict_proba(X_valid)[:, 1]
        pred_valid = (proba_valid >= 0.50).astype(int)

        try:
            avg_precision = average_precision_score(y_valid, proba_valid)
        except ValueError:
            avg_precision = np.nan

        try:
            roc_auc = roc_auc_score(y_valid, proba_valid)
        except ValueError:
            roc_auc = np.nan

        metrics = {
            "product": product_name,
            "target_col": target_col,
            "status": "trained",
            "model": model_name,
            "n_total": len(y),
            "positive_customers": int(y.sum()),
            "positive_rate": float(y.mean()),
            "train_rows": len(X_train),
            "valid_rows": len(X_valid),
            "valid_positive_customers": int(y_valid.sum()),
            "valid_avg_precision": avg_precision,
            "valid_roc_auc": roc_auc,
            "valid_precision_at_050": precision_score(y_valid, pred_valid, zero_division=0),
            "valid_recall_at_050": recall_score(y_valid, pred_valid, zero_division=0),
            "n_numeric_features": len(numeric_cols),
            "n_categorical_features": len(categorical_cols),
            **baseline}

        fitted_models.append((metrics, pipe))

    # select best candidate by average precision; use ROC-AUC as tie breaker
    # in lead generation, we care about ranking good leads near the top -
    # Average precision rewards models that assign high probabilities to true buyers
    best_metrics, best_pipe = max(
        fitted_models,
        key=lambda item: (
            np.nan_to_num(item[0]["valid_avg_precision"], nan=-1),
            np.nan_to_num(item[0]["valid_roc_auc"], nan=-1)))

    # refit the best model on all rows before scoring all customers.
    best_X = df.drop(columns=drop_cols, errors="ignore").copy()
    best_pipe.fit(best_X, y)

    return best_pipe, best_metrics

## Generate Recommendations for One productg group

In [10]:
""" For each target:
- train the best model
- score all customers
- keep only customers who have not bought the product yet
- create a long recommendation table """

def build_recommendations_for_group(
    df: pd.DataFrame,
    target_prefix: str,
    product_group_label: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    target_cols = get_target_cols(df, target_prefix)
    all_recommendations = []
    model_registry_rows = []

    for target_col in target_cols:
        model, metrics = train_one_product_model(
            df=df,
            target_col=target_col,
            all_target_cols=target_cols,
            product_prefix=target_prefix)
        metrics["product_group"] = product_group_label
        model_registry_rows.append(metrics)

        if model is None:
            continue

        product_name = target_col.replace(target_prefix, "")
        drop_cols = base_columns_to_drop(df) + target_cols
        X_all = df.drop(columns=drop_cols, errors="ignore").copy()

        # rredict probability for every customer
        proba_all = model.predict_proba(X_all)[:, 1]

        # recommend only to customers who have not bought that product group yet
        not_bought_mask = df[target_col].astype(int).eq(0)

        recs = pd.DataFrame({
            "ID_Customer": df.loc[not_bought_mask, "ID_Customer"].values,
            "product_group": product_group_label,
            "recommended_product": product_name,
            "model_probability": proba_all[not_bought_mask],
            "source_target_col": target_col})

        all_recommendations.append(recs)

    if all_recommendations:
        recommendations_long = pd.concat(all_recommendations, ignore_index=True)
    else:
        recommendations_long = pd.DataFrame(
            columns=[
                "ID_Customer",
                "product_group",
                "recommended_product",
                "model_probability",
                "source_target_col"])

    model_registry = pd.DataFrame(model_registry_rows)
    return recommendations_long, model_registry

## Train Product Group 1 Recommendation Models

In [11]:
""" Product group 1 is a higher-level recommendation layer.
This is useful when sales teams need broader guidance, such as:
- recommend a product category
- identify cross-sell potential
- create simple sales plays """

recs_grp1_long, registry_grp1 = build_recommendations_for_group(
    df=df_model_grp1,
    target_prefix="purchase_flag_grp1_",
    product_group_label="grp1")

print("Group 1 recommendations:", recs_grp1_long.shape)
print("Group 1 model registry:", registry_grp1.shape)

display(registry_grp1.sort_values("valid_avg_precision", ascending=False))
display(recs_grp1_long.head())


Group 1 recommendations: (4232, 5)
Group 1 model registry: (6, 19)


,product,target_col,status,model,n_total,positive_customers,positive_rate,train_rows,valid_rows,valid_positive_customers,valid_avg_precision,valid_roc_auc,valid_precision_at_050,valid_recall_at_050,n_numeric_features,n_categorical_features,baseline_avg_precision,baseline_roc_auc,product_group
1,Core Products,purchase_flag_grp1_Core Products,trained,LogisticRegression,1993,1622,0.8138,1594,399,325,0.9675,0.8696,0.9345,0.7908,40,3,0.8145,0.5000,grp1
0,Automation Modules,purchase_flag_grp1_Automation Modules,trained,RandomForest,1993,1656,0.8309,1594,399,332,0.9466,0.7845,0.9015,0.8825,40,3,0.8321,0.5000,grp1
5,Service Contracts,purchase_flag_grp1_Service Contracts,trained,LogisticRegression,1993,1174,0.5891,1594,399,235,0.9052,0.8660,0.8259,0.7872,40,3,0.5890,0.5000,grp1
2,Maintenance Kits,purchase_flag_grp1_Maintenance Kits,trained,LogisticRegression,1993,1310,0.6573,1594,399,262,0.8470,0.7318,0.7792,0.6870,40,3,0.6566,0.5000,grp1
3,Premium Products,purchase_flag_grp1_Premium Products,trained,LogisticRegression,1993,1055,0.5294,1594,399,211,0.7834,0.7492,0.7157,0.6682,40,3,0.5288,0.5000,grp1
4,Safety Solutions,purchase_flag_grp1_Safety Solutions,trained,LogisticRegression,1993,909,0.4561,1594,399,182,0.7760,0.7997,0.6667,0.7473,40,3,0.4561,0.5000,grp1


,ID_Customer,product_group,recommended_product,model_probability,source_target_col
0,C00003,grp1,Automation Modules,0.4632,purchase_flag_grp1_Automation Modules
1,C00014,grp1,Automation Modules,0.1942,purchase_flag_grp1_Automation Modules
2,C00021,grp1,Automation Modules,0.3737,purchase_flag_grp1_Automation Modules
3,C00026,grp1,Automation Modules,0.3569,purchase_flag_grp1_Automation Modules
4,C00029,grp1,Automation Modules,0.4936,purchase_flag_grp1_Automation Modules


## Train Product Group 3 Recommendation Models

In [12]:
""" Product group 3 is more granular.
This is more actionable but also harder:
- more targets
- lower positive rates per product
- stronger class imbalance
For that reason, some rare targets may be skipped if they do not have enough positive customers """

recs_grp3_long, registry_grp3 = build_recommendations_for_group(
    df=df_model_grp3,
    target_prefix="purchase_flag_grp3_",
    product_group_label="grp3")

print("Group 3 recommendations:", recs_grp3_long.shape)
print("Group 3 model registry:", registry_grp3.shape)

display(registry_grp3.sort_values("valid_avg_precision", ascending=False).head(20))
display(recs_grp3_long.head())

Group 3 recommendations: (56758, 5)
Group 3 model registry: (35, 19)


,product,target_col,status,model,n_total,positive_customers,positive_rate,train_rows,valid_rows,valid_positive_customers,valid_avg_precision,valid_roc_auc,valid_precision_at_050,valid_recall_at_050,n_numeric_features,n_categorical_features,baseline_avg_precision,baseline_roc_auc,product_group
9,Core Basic Type 2,purchase_flag_grp3_Core Basic Type 2,trained,LogisticRegression,1993,678,0.3402,1594,399,136,0.5241,0.6985,0.4971,0.6397,40,3,0.3409,0.5000,grp3
7,Core Advanced Type 2,purchase_flag_grp3_Core Advanced Type 2,trained,LogisticRegression,1993,695,0.3487,1594,399,139,0.5240,0.6765,0.5000,0.6187,40,3,0.3484,0.5000,grp3
15,Installation Type 1,purchase_flag_grp3_Installation Type 1,trained,LogisticRegression,1993,375,0.1882,1594,399,75,0.5061,0.8237,0.3971,0.7200,40,3,0.1880,0.5000,grp3
31,Sensors Type 1,purchase_flag_grp3_Sensors Type 1,trained,LogisticRegression,1993,669,0.3357,1594,399,134,0.4998,0.6596,0.4492,0.6269,40,3,0.3358,0.5000,grp3
6,Core Advanced Type 1,purchase_flag_grp3_Core Advanced Type 1,trained,LogisticRegression,1993,609,0.3056,1594,399,122,0.4930,0.7028,0.4489,0.6475,40,3,0.3058,0.5000,grp3
4,Controllers Type 1,purchase_flag_grp3_Controllers Type 1,trained,RandomForest,1993,684,0.3432,1594,399,137,0.4637,0.6487,0.4427,0.4234,40,3,0.3434,0.5000,grp3
32,Sensors Type 2,purchase_flag_grp3_Sensors Type 2,trained,LogisticRegression,1993,586,0.2940,1594,399,117,0.4501,0.6772,0.4057,0.6068,40,3,0.2932,0.5000,grp3
8,Core Basic Type 1,purchase_flag_grp3_Core Basic Type 1,trained,LogisticRegression,1993,601,0.3016,1594,399,120,0.4180,0.6709,0.4239,0.6500,40,3,0.3008,0.5000,grp3
19,Personal Safety Type 1,purchase_flag_grp3_Personal Safety Type 1,trained,RandomForest,1993,360,0.1806,1594,399,72,0.3765,0.6905,0.4133,0.4306,40,3,0.1805,0.5000,grp3
17,Monitoring Type 1,purchase_flag_grp3_Monitoring Type 1,trained,LogisticRegression,1993,459,0.2303,1594,399,92,0.3689,0.6872,0.3613,0.6087,40,3,0.2306,0.5000,grp3


,ID_Customer,product_group,recommended_product,model_probability,source_target_col
0,C00001,grp3,Calibration Kits Type 1,0.4659,purchase_flag_grp3_Calibration Kits Type 1
1,C00002,grp3,Calibration Kits Type 1,0.6748,purchase_flag_grp3_Calibration Kits Type 1
2,C00003,grp3,Calibration Kits Type 1,0.6673,purchase_flag_grp3_Calibration Kits Type 1
3,C00004,grp3,Calibration Kits Type 1,0.5388,purchase_flag_grp3_Calibration Kits Type 1
4,C00005,grp3,Calibration Kits Type 1,0.1030,purchase_flag_grp3_Calibration Kits Type 1


## Review Model Performance

In [13]:
""" model registry helps us understand which product targets are reliable.
Useful columns:
- positive_rate: how common the product is
- valid_avg_precision: ranking quality
- baseline_avg_precision: popularity baseline
- valid_roc_auc: separation between buyers and non-buyers
- status: whether the model was trained or skipped """

model_registry_all = pd.concat([registry_grp1, registry_grp3], ignore_index=True)

model_registry_all["avg_precision_lift_vs_baseline"] = (
    model_registry_all["valid_avg_precision"] / model_registry_all["baseline_avg_precision"]
).replace([np.inf, -np.inf], np.nan)

display(
    model_registry_all
    .sort_values(["status", "valid_avg_precision"], ascending=[True, False])
    .head(30))

display(
    model_registry_all
    .groupby(["product_group", "status"], as_index=False)
    .agg(
        targets=("target_col", "count"),
        avg_positive_rate=("positive_rate", "mean"),
        avg_valid_avg_precision=("valid_avg_precision", "mean"),
        avg_baseline_avg_precision=("baseline_avg_precision", "mean"),
        avg_precision_lift=("avg_precision_lift_vs_baseline", "mean")))

,product,target_col,status,model,n_total,positive_customers,positive_rate,train_rows,valid_rows,valid_positive_customers,valid_avg_precision,valid_roc_auc,valid_precision_at_050,valid_recall_at_050,n_numeric_features,n_categorical_features,baseline_avg_precision,baseline_roc_auc,product_group,avg_precision_lift_vs_baseline
1,Core Products,purchase_flag_grp1_Core Products,trained,LogisticRegression,1993,1622,0.8138,1594,399,325,0.9675,0.8696,0.9345,0.7908,40,3,0.8145,0.5000,grp1,1.1878
0,Automation Modules,purchase_flag_grp1_Automation Modules,trained,RandomForest,1993,1656,0.8309,1594,399,332,0.9466,0.7845,0.9015,0.8825,40,3,0.8321,0.5000,grp1,1.1376
5,Service Contracts,purchase_flag_grp1_Service Contracts,trained,LogisticRegression,1993,1174,0.5891,1594,399,235,0.9052,0.8660,0.8259,0.7872,40,3,0.5890,0.5000,grp1,1.5369
2,Maintenance Kits,purchase_flag_grp1_Maintenance Kits,trained,LogisticRegression,1993,1310,0.6573,1594,399,262,0.8470,0.7318,0.7792,0.6870,40,3,0.6566,0.5000,grp1,1.2899
3,Premium Products,purchase_flag_grp1_Premium Products,trained,LogisticRegression,1993,1055,0.5294,1594,399,211,0.7834,0.7492,0.7157,0.6682,40,3,0.5288,0.5000,grp1,1.4814
4,Safety Solutions,purchase_flag_grp1_Safety Solutions,trained,LogisticRegression,1993,909,0.4561,1594,399,182,0.7760,0.7997,0.6667,0.7473,40,3,0.4561,0.5000,grp1,1.7011
15,Core Basic Type 2,purchase_flag_grp3_Core Basic Type 2,trained,LogisticRegression,1993,678,0.3402,1594,399,136,0.5241,0.6985,0.4971,0.6397,40,3,0.3409,0.5000,grp3,1.5377
13,Core Advanced Type 2,purchase_flag_grp3_Core Advanced Type 2,trained,LogisticRegression,1993,695,0.3487,1594,399,139,0.5240,0.6765,0.5000,0.6187,40,3,0.3484,0.5000,grp3,1.5040
21,Installation Type 1,purchase_flag_grp3_Installation Type 1,trained,LogisticRegression,1993,375,0.1882,1594,399,75,0.5061,0.8237,0.3971,0.7200,40,3,0.1880,0.5000,grp3,2.6927
37,Sensors Type 1,purchase_flag_grp3_Sensors Type 1,trained,LogisticRegression,1993,669,0.3357,1594,399,134,0.4998,0.6596,0.4492,0.6269,40,3,0.3358,0.5000,grp3,1.4884


,product_group,status,targets,avg_positive_rate,avg_valid_avg_precision,avg_baseline_avg_precision,avg_precision_lift
0,grp1,trained,6,0.6461,0.8709,0.6462,1.3891
1,grp3,trained,35,0.1863,0.3082,0.1861,1.7546


## Rank Recommendations

In [14]:
# combine grp 1 and grp 3 recommendations
# ranking is based first on model probability. later we combine it with business signals to create a lead score

recommendations_long = pd.concat([recs_grp1_long, recs_grp3_long], ignore_index=True)

recommendations_long = recommendations_long.sort_values(
    ["ID_Customer", "model_probability"],
    ascending=[True, False])

# ranking among customers
recommendations_long["model_rank_per_customer"] = (
    recommendations_long
    .groupby("ID_Customer")
    .cumcount()
    + 1)

recommendations_topk_model = recommendations_long[
    recommendations_long["model_rank_per_customer"] <= TOP_K
].copy()

print("All recommendation rows:", recommendations_long.shape)
print("Top-K model recommendation rows:", recommendations_topk_model.shape)

display(recommendations_topk_model.head(20))

All recommendation rows: (60990, 6)
Top-K model recommendation rows: (5979, 6)


,ID_Customer,product_group,recommended_product,model_probability,source_target_col,model_rank_per_customer
44923,C00001,grp3,Premium Standard Type 1,0.6251,purchase_flag_grp3_Premium Standard Type 1,1
32895,C00001,grp3,Monitoring Type 2,0.5645,purchase_flag_grp3_Monitoring Type 2,2
2329,C00001,grp1,Safety Solutions,0.5630,purchase_flag_grp1_Safety Solutions,3
13810,C00002,grp3,Core Advanced Type 1,0.7298,purchase_flag_grp3_Core Advanced Type 1,1
4233,C00002,grp3,Calibration Kits Type 1,0.6748,purchase_flag_grp3_Calibration Kits Type 1,2
708,C00002,grp1,Maintenance Kits,0.6422,purchase_flag_grp1_Maintenance Kits,3
37994,C00003,grp3,Premium Custom Type 1,0.7909,purchase_flag_grp3_Premium Custom Type 1,1
24355,C00003,grp3,Extended Support Type 2,0.7898,purchase_flag_grp3_Extended Support Type 2,2
54962,C00003,grp3,Sensors Type 1,0.7467,purchase_flag_grp3_Sensors Type 1,3
37995,C00004,grp3,Premium Custom Type 1,0.5628,purchase_flag_grp3_Premium Custom Type 1,1


## Expected order value and expected revenue
The source data has no order or invoice ID. The code therefore treats a **customer-month-product-group** as the closest available order event. 
Because recommended products have not previously been bought by the customer, there is no customer-product order history. Expected order value is therefore estimated as follows:

1. Calculate average historical event value by customer segment and product
2. Shrink sparse segment-product averages toward the product-wide average
3. Adjust for the customer's general buying power relative to their segment
4. Cap the buying-power multiplier to avoid unrealistic extrapolation

The result is robust enough for prioritization while retaining transparent support and fallback fields for validation.

In [15]:
# Prepare a historical value-event table.
# Prefer a true order/invoice identifier when available; otherwise use customer-month.

sales_for_value = df_sales_full.copy()
sales_for_value["Sales"] = pd.to_numeric(sales_for_value["Sales"], errors="coerce")
sales_for_value = sales_for_value[sales_for_value["Sales"].gt(0)].copy()

if "Units" in sales_for_value.columns:
    sales_for_value["Units"] = pd.to_numeric(sales_for_value["Units"], errors="coerce")
    sales_for_value = sales_for_value[sales_for_value["Units"].fillna(0).gt(0)].copy()

order_id_candidates = [
    "ID_Order", "Order_ID", "order_id", "Invoice_ID", "invoice_id", "Transaction_ID"]
order_id_col = next((c for c in order_id_candidates if c in sales_for_value.columns), None)

if order_id_col is not None:
    sales_for_value["_value_event_id"] = sales_for_value[order_id_col].astype("string")
    VALUE_EVENT_GRAIN = order_id_col
else:
    if "Date" not in sales_for_value.columns:
        raise KeyError("Expected either an order/invoice ID or a Date column in df_sales_full.")
    sales_for_value["Date"] = pd.to_datetime(sales_for_value["Date"], errors="coerce")
    sales_for_value["_value_event_id"] = sales_for_value["Date"].dt.to_period("M").astype("string")
    VALUE_EVENT_GRAIN = "customer_month_product_group_proxy"

# Customer segment is the preferred peer group. Cohort is a fallback.
customer_profiles = df_customers[["ID_Customer"]].drop_duplicates().copy()

if "cohort_ML" in df_customers.columns:
    customer_profiles = customer_profiles.merge(
        df_customers[["ID_Customer", "cohort_ML"]],
        on="ID_Customer",
        how="left",
        validate="one_to_one")

if "Customer_Segment" in sales_for_value.columns:
    latest_customer_segment = (
        sales_for_value
        .sort_values(["ID_Customer", "_value_event_id"])
        .dropna(subset=["Customer_Segment"])
        .drop_duplicates("ID_Customer", keep="last")
        [["ID_Customer", "Customer_Segment"]])
    customer_profiles = customer_profiles.merge(
        latest_customer_segment,
        on="ID_Customer",
        how="left",
        validate="one_to_one")

if "Customer_Segment" in customer_profiles.columns:
    customer_profiles["value_segment"] = customer_profiles["Customer_Segment"]
else:
    customer_profiles["value_segment"] = np.nan

if "cohort_ML" in customer_profiles.columns:
    customer_profiles["value_segment"] = customer_profiles["value_segment"].fillna(
        customer_profiles["cohort_ML"])

customer_profiles["value_segment"] = customer_profiles["value_segment"].fillna("Unknown").astype(str)


def build_product_value_events(
    sales: pd.DataFrame,
    product_col: str,
    product_group_label: str,
) -> pd.DataFrame:
    """Aggregate sales to the best available order-event grain for one hierarchy."""
    required = {"ID_Customer", "_value_event_id", "Sales", product_col}
    missing = required.difference(sales.columns)
    if missing:
        raise KeyError(f"Missing columns for {product_group_label}: {sorted(missing)}")

    events = (
        sales
        .dropna(subset=["ID_Customer", "_value_event_id", product_col])
        .groupby(["ID_Customer", "_value_event_id", product_col], as_index=False)
        .agg(order_value=("Sales", "sum"))
        .rename(columns={product_col: "recommended_product"}))

    events["product_group"] = product_group_label
    return events


product_hierarchy = {
    "grp1": "Prd_Grp_1",
    "grp3": "Prd_Grp_3"}

value_events = pd.concat(
    [
        build_product_value_events(sales_for_value, product_col, group_label)
        for group_label, product_col in product_hierarchy.items()],
    ignore_index=True)

value_events = value_events.merge(
    customer_profiles[["ID_Customer", "value_segment"]],
    on="ID_Customer",
    how="left",
    validate="many_to_one")
value_events["value_segment"] = value_events["value_segment"].fillna("Unknown")


def winsorize_if_supported(series: pd.Series) -> pd.Series:
    """Cap extreme event values only when the product has enough observations."""
    if series.notna().sum() < 20:
        return series
    lower = series.quantile(WINSOR_LIMITS[0])
    upper = series.quantile(WINSOR_LIMITS[1])
    return series.clip(lower=lower, upper=upper)


value_events["order_value_capped"] = (
    value_events
    .groupby(["product_group", "recommended_product"])["order_value"]
    .transform(winsorize_if_supported))

# product-wide fallback
product_value_stats = (
    value_events
    .groupby(["product_group", "recommended_product"], as_index=False)
    .agg(
        product_mean_order_value=("order_value_capped", "mean"),
        product_median_order_value=("order_value_capped", "median"),
        product_value_events=("order_value_capped", "size")))

# more specific peer-group estimate
segment_product_value_stats = (
    value_events
    .groupby(["value_segment", "product_group", "recommended_product"], as_index=False)
    .agg(
        segment_product_mean_order_value=("order_value_capped", "mean"),
        segment_product_median_order_value=("order_value_capped", "median"),
        segment_product_value_events=("order_value_capped", "size")))

# customer buying-power multiplier based on average total event value across all products
customer_value_events = (
    sales_for_value
    .groupby(["ID_Customer", "_value_event_id"], as_index=False)
    .agg(customer_event_value=("Sales", "sum")))

customer_value_profiles = (
    customer_value_events
    .groupby("ID_Customer", as_index=False)
    .agg(
        customer_mean_event_value=("customer_event_value", "mean"),
        customer_median_event_value=("customer_event_value", "median"),
        customer_value_events=("customer_event_value", "size"))
    .merge(
        customer_profiles[["ID_Customer", "value_segment"]],
        on="ID_Customer",
        how="left",
        validate="one_to_one"))

segment_customer_benchmark = (
    customer_value_profiles
    .groupby("value_segment", as_index=False)
    .agg(segment_median_customer_event_value=("customer_mean_event_value", "median")))

customer_value_profiles = customer_value_profiles.merge(
    segment_customer_benchmark,
    on="value_segment",
    how="left",
    validate="many_to_one")

customer_value_profiles["customer_value_multiplier_raw"] = (
    customer_value_profiles["customer_mean_event_value"]
    / customer_value_profiles["segment_median_customer_event_value"].replace(0, np.nan))

customer_value_profiles["customer_value_multiplier"] = (
    customer_value_profiles["customer_value_multiplier_raw"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(1.0)
    .clip(*CUSTOMER_VALUE_MULTIPLIER_BOUNDS))

print("Value-event grain:", VALUE_EVENT_GRAIN)
print("Value-event rows:", value_events.shape)
display(product_value_stats.head())
display(segment_product_value_stats.head())
display(customer_value_profiles.head())

Value-event grain: customer_month_product_group_proxy
Value-event rows: (119768, 7)


,product_group,recommended_product,product_mean_order_value,product_median_order_value,product_value_events
0,grp1,Automation Modules,"4,339.6306","2,489.0200",12949
1,grp1,Core Products,595.7803,357.4700,12374
2,grp1,Maintenance Kits,929.1495,574.0900,8690
3,grp1,Premium Products,"2,190.6499","1,312.2200",6555
4,grp1,Safety Solutions,505.4395,312.6500,5642


,value_segment,product_group,recommended_product,segment_product_mean_order_value,segment_product_median_order_value,segment_product_value_events
0,Distributor,grp1,Automation Modules,"4,974.1448","3,361.8400",1260
1,Distributor,grp1,Core Products,866.1408,623.6750,1692
2,Distributor,grp1,Maintenance Kits,"1,349.5839",957.6100,1363
3,Distributor,grp1,Premium Products,"3,108.2803","2,185.5400",1405
4,Distributor,grp1,Safety Solutions,757.2329,643.1000,507


,ID_Customer,customer_mean_event_value,customer_median_event_value,customer_value_events,value_segment,segment_median_customer_event_value,customer_value_multiplier_raw,customer_value_multiplier
0,C00001,"6,279.7074","5,759.4200",23,Distributor,"2,719.2308",2.3094,2.0000
1,C00002,592.3262,483.4050,8,SMB,919.5186,0.6442,0.6442
2,C00003,"6,431.0258","5,259.4000",31,SMB,919.5186,6.9939,2.0000
3,C00004,"5,004.9854","4,912.7450",26,Mid-Market,"2,157.5030",2.3198,2.0000
4,C00005,"2,301.3206","2,211.2700",17,SMB,919.5186,2.5027,2.0000


In [16]:
def add_expected_value_to_recommendations(recommendations: pd.DataFrame) -> pd.DataFrame:
    """attach transparent order-value estimates and probability-weighted revenue"""
    original_columns = recommendations.columns.tolist()

    enriched = recommendations.merge(
        customer_profiles[["ID_Customer", "value_segment"]],
        on="ID_Customer",
        how="left",
        validate="many_to_one")

    enriched = enriched.merge(
        segment_product_value_stats,
        on=["value_segment", "product_group", "recommended_product"],
        how="left",
        validate="many_to_one")

    enriched = enriched.merge(
        product_value_stats,
        on=["product_group", "recommended_product"],
        how="left",
        validate="many_to_one")

    enriched = enriched.merge(
        customer_value_profiles[[
            "ID_Customer",
            "customer_mean_event_value",
            "customer_value_events",
            "customer_value_multiplier"]],
        on="ID_Customer",
        how="left",
        validate="many_to_one")

    support = enriched["segment_product_value_events"].fillna(0)
    enriched["value_shrinkage_weight"] = support / (
        support + VALUE_SHRINKAGE_STRENGTH)

    # if the segment-product combination is unavailable, its weight is zero and
    # the estimate falls back to the product-wide mean
    segment_mean = enriched["segment_product_mean_order_value"].fillna(
        enriched["product_mean_order_value"])

    enriched["expected_order_value_before_customer_adjustment"] = (
        enriched["value_shrinkage_weight"] * segment_mean
        + (1 - enriched["value_shrinkage_weight"])
        * enriched["product_mean_order_value"])

    global_fallback = value_events["order_value_capped"].mean()
    enriched["expected_order_value_before_customer_adjustment"] = (
        enriched["expected_order_value_before_customer_adjustment"]
        .fillna(global_fallback))

    enriched["customer_value_multiplier"] = (
        enriched["customer_value_multiplier"].fillna(1.0))

    enriched["expected_order_value"] = (
        enriched["expected_order_value_before_customer_adjustment"]
        * enriched["customer_value_multiplier"]
    ).clip(lower=0)

    enriched["expected_revenue"] = (
        enriched["model_probability"].clip(0, 1)
        * enriched["expected_order_value"])

    # Keep a deliberately explicit alias while the probability is an affinity score.
    enriched["expected_revenue_proxy"] = enriched["expected_revenue"]
    enriched["expected_revenue_is_forecast"] = (
        MODEL_PROBABILITY_IS_CALIBRATED_FUTURE_PROBABILITY)
    enriched["probability_semantics"] = np.where(
        MODEL_PROBABILITY_IS_CALIBRATED_FUTURE_PROBABILITY,
        "calibrated future-purchase probability",
        "historical product-affinity score")

    enriched["order_value_estimation_method"] = np.select(
        [
            support.ge(VALUE_SHRINKAGE_STRENGTH),
            support.gt(0),
            enriched["product_mean_order_value"].notna()],
        [
            "supported segment-product estimate with shrinkage",
            "low-support segment-product estimate with shrinkage",
            "product-wide fallback"],
        default="global fallback")

    added_columns = [c for c in enriched.columns if c not in original_columns]
    return enriched[original_columns + added_columns]


recommendations_long = add_expected_value_to_recommendations(recommendations_long)

# recreate Top-K after adding the value columns so downstream exports retain them
recommendations_topk_model = recommendations_long[
    recommendations_long["model_rank_per_customer"] <= TOP_K
].copy()

display(recommendations_long[[
    "ID_Customer",
    "value_segment",
    "product_group",
    "recommended_product",
    "model_probability",
    "expected_order_value",
    "expected_revenue",
    "segment_product_value_events",
    "customer_value_multiplier",
    "order_value_estimation_method",
    "probability_semantics"]].head(20))

,ID_Customer,value_segment,product_group,recommended_product,model_probability,expected_order_value,expected_revenue,segment_product_value_events,customer_value_multiplier,order_value_estimation_method,probability_semantics
0,C00001,Distributor,grp3,Premium Standard Type 1,0.6251,"2,803.9819","1,752.6986",137,2.0000,supported segment-product estimate with shrinkage,historical product-affinity score
1,C00001,Distributor,grp3,Monitoring Type 2,0.5645,"5,033.0853","2,841.1890",126,2.0000,supported segment-product estimate with shrinkage,historical product-affinity score
2,C00001,Distributor,grp1,Safety Solutions,0.5630,"1,486.3325",836.8758,507,2.0000,supported segment-product estimate with shrinkage,historical product-affinity score
3,C00001,Distributor,grp3,Installation Type 2,0.5362,"8,132.8978","4,360.8969",81,2.0000,supported segment-product estimate with shrinkage,historical product-affinity score
4,C00001,Distributor,grp3,Training Type 2,0.5139,"11,198.7280","5,755.4957",233,2.0000,supported segment-product estimate with shrinkage,historical product-affinity score
5,C00001,Distributor,grp3,Calibration Kits Type 2,0.4949,"2,363.5496","1,169.8211",305,2.0000,supported segment-product estimate with shrinkage,historical product-affinity score
6,C00001,Distributor,grp3,Premium Plus Type 2,0.4949,"4,609.6710","2,281.2755",295,2.0000,supported segment-product estimate with shrinkage,historical product-affinity score
7,C00001,Distributor,grp1,Service Contracts,0.4669,"12,633.2832","5,897.8985",793,2.0000,supported segment-product estimate with shrinkage,historical product-affinity score
8,C00001,Distributor,grp3,Calibration Kits Type 1,0.4659,"1,902.0879",886.1823,236,2.0000,supported segment-product estimate with shrinkage,historical product-affinity score
9,C00001,Distributor,grp3,Core Basic Type 1,0.4406,"1,355.3503",597.2327,375,2.0000,supported segment-product estimate with shrinkage,historical product-affinity score


In [17]:
# Quality assessment expected revenue
expected_value_quality = pd.DataFrame({
    "metric": [
        "recommendation rows",
        "missing expected order value",
        "non-positive expected order value",
        "missing expected revenue",
        "segment-product fallback share",
        "median segment-product support"],
    "value": [
        len(recommendations_long),
        recommendations_long["expected_order_value"].isna().sum(),
        recommendations_long["expected_order_value"].le(0).sum(),
        recommendations_long["expected_revenue"].isna().sum(),
        recommendations_long["segment_product_value_events"].fillna(0).eq(0).mean(),
        recommendations_long["segment_product_value_events"].median()]})

display(expected_value_quality)

assert recommendations_long["expected_order_value"].notna().all()
assert recommendations_long["expected_order_value"].gt(0).all()
assert recommendations_long["expected_revenue"].notna().all()
assert np.allclose(
    recommendations_long["expected_revenue"],
    recommendations_long["model_probability"]
    * recommendations_long["expected_order_value"])

if not MODEL_PROBABILITY_IS_CALIBRATED_FUTURE_PROBABILITY:
    print(
        "WARNING: expected_revenue is a propensity-weighted opportunity proxy, "
        "because the current model is not a calibrated future-purchase model.")

,metric,value
0,recommendation rows,"60,990.0000"
1,missing expected order value,0.0000
2,non-positive expected order value,0.0000
3,missing expected revenue,0.0000
4,segment-product fallback share,0.0000
5,median segment-product support,367.0000


## Create wide Recommendation Tables

In [18]:
def make_topk_wide(recs_long: pd.DataFrame, product_group: str, top_k: int = TOP_K) -> pd.DataFrame:
    part = recs_long[recs_long["product_group"] == product_group].copy()

    if part.empty:
        return pd.DataFrame(columns=["ID_Customer"])

    part = part.sort_values(
        ["ID_Customer", "model_probability"],
        ascending=[True, False])
    part["rank"] = part.groupby("ID_Customer").cumcount() + 1
    part = part[part["rank"] <= top_k].copy()

    metric_cols = [
        "recommended_product",
        "model_probability",
        "expected_order_value",
        "expected_revenue",
        "order_value_estimation_method",
        "rank"]
    metric_cols = [c for c in metric_cols if c in part.columns]

    wide_parts = []
    for k in range(1, top_k + 1):
        rename_map = {
            "recommended_product": f"reco_top{k}_{product_group}_product",
            "model_probability": f"reco_top{k}_{product_group}_probability",
            "expected_order_value": f"reco_top{k}_{product_group}_expected_order_value",
            "expected_revenue": f"reco_top{k}_{product_group}_expected_revenue",
            "order_value_estimation_method": f"reco_top{k}_{product_group}_value_method",
            "rank": f"reco_top{k}_{product_group}_rank"}
        kth = (
            part[part["rank"] == k][["ID_Customer"] + metric_cols]
            .rename(columns=rename_map))
        wide_parts.append(kth)

    wide = wide_parts[0]
    for kth in wide_parts[1:]:
        wide = wide.merge(kth, on="ID_Customer", how="outer")

    return wide


recos_grp1_top3_wide = make_topk_wide(recommendations_long, "grp1", TOP_K)
recos_grp3_top3_wide = make_topk_wide(recommendations_long, "grp3", TOP_K)

display(recos_grp1_top3_wide.head())
display(recos_grp3_top3_wide.head())

,ID_Customer,reco_top1_grp1_product,reco_top1_grp1_probability,reco_top1_grp1_expected_order_value,reco_top1_grp1_expected_revenue,reco_top1_grp1_value_method,reco_top1_grp1_rank,reco_top2_grp1_product,reco_top2_grp1_probability,reco_top2_grp1_expected_order_value,reco_top2_grp1_expected_revenue,reco_top2_grp1_value_method,reco_top2_grp1_rank,reco_top3_grp1_product,reco_top3_grp1_probability,reco_top3_grp1_expected_order_value,reco_top3_grp1_expected_revenue,reco_top3_grp1_value_method,reco_top3_grp1_rank
0,C00001,Safety Solutions,0.5630,"1,486.3325",836.8758,supported segment-product estimate with shrinkage,1,Service Contracts,0.4669,"12,633.2832","5,897.8985",supported segment-product estimate with shrinkage,2.0000,NaN,NaN,NaN,NaN,NaN,NaN
1,C00002,Maintenance Kits,0.6422,259.2784,166.4986,supported segment-product estimate with shrinkage,1,Service Contracts,0.3371,"1,319.7295",444.8734,supported segment-product estimate with shrinkage,2.0000,NaN,NaN,NaN,NaN,NaN,NaN
2,C00003,Automation Modules,0.4632,"2,972.6059","1,377.0508",supported segment-product estimate with shrinkage,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,C00004,Premium Products,0.3831,"3,803.6985","1,457.0397",supported segment-product estimate with shrinkage,1,Safety Solutions,0.2818,"1,382.3802",389.5469,supported segment-product estimate with shrinkage,2.0000,Service Contracts,0.1430,"8,911.7568","1,274.2352",supported segment-product estimate with shrinkage,3.0000
4,C00005,Safety Solutions,0.3228,545.6936,176.1258,supported segment-product estimate with shrinkage,1,Maintenance Kits,0.2681,805.0000,215.7820,supported segment-product estimate with shrinkage,2.0000,NaN,NaN,NaN,NaN,NaN,NaN


,ID_Customer,reco_top1_grp3_product,reco_top1_grp3_probability,reco_top1_grp3_expected_order_value,reco_top1_grp3_expected_revenue,reco_top1_grp3_value_method,reco_top1_grp3_rank,reco_top2_grp3_product,reco_top2_grp3_probability,reco_top2_grp3_expected_order_value,reco_top2_grp3_expected_revenue,reco_top2_grp3_value_method,reco_top2_grp3_rank,reco_top3_grp3_product,reco_top3_grp3_probability,reco_top3_grp3_expected_order_value,reco_top3_grp3_expected_revenue,reco_top3_grp3_value_method,reco_top3_grp3_rank
0,C00001,Premium Standard Type 1,0.6251,"2,803.9819","1,752.6986",supported segment-product estimate with shrinkage,1,Monitoring Type 2,0.5645,"5,033.0853","2,841.1890",supported segment-product estimate with shrinkage,2,Installation Type 2,0.5362,"8,132.8978","4,360.8969",supported segment-product estimate with shrinkage,3
1,C00002,Core Advanced Type 1,0.7298,180.9113,132.0343,supported segment-product estimate with shrinkage,1,Calibration Kits Type 1,0.6748,205.0932,138.3905,supported segment-product estimate with shrinkage,2,Sensors Type 2,0.6217,647.0804,402.2769,supported segment-product estimate with shrinkage,3
2,C00003,Premium Custom Type 1,0.7909,"2,315.4905","1,831.2782",supported segment-product estimate with shrinkage,1,Extended Support Type 2,0.7898,"3,883.1339","3,066.9553",supported segment-product estimate with shrinkage,2,Sensors Type 1,0.7467,"2,573.0797","1,921.2178",supported segment-product estimate with shrinkage,3
3,C00004,Premium Custom Type 1,0.5628,"4,066.6452","2,288.5251",supported segment-product estimate with shrinkage,1,Calibration Kits Type 1,0.5388,"1,026.9690",553.3309,supported segment-product estimate with shrinkage,2,Core Advanced Type 2,0.4544,"1,151.6357",523.3357,supported segment-product estimate with shrinkage,3
4,C00005,Training Type 2,0.7913,"2,559.6315","2,025.3860",supported segment-product estimate with shrinkage,1,Training Type 1,0.7586,"4,633.4868","3,515.1846",supported segment-product estimate with shrinkage,2,Extended Support Type 2,0.5815,"3,883.1339","2,258.2355",supported segment-product estimate with shrinkage,3


## Build Lead Scoring Dataset

In [19]:
""" A product recommendation probability alone is not enough.
Sales teams also care about: customer value, recent activity, churn or inactivity risk, cohort context.
The lead score combines model probability with business value and customer engagement. """

customer_context_cols = [
    "ID_Customer",
    "cohort_ML",
    "cluster",
    "Sales",
    "historical_ltv",
    "avg_monthly_revenue",
    "active_months",
    "months_since_last_purchase",
    "activity_ratio_lifetime",
    "avg_inactive_gap_months",
    "prd_grp_1_coverage_pct",
    "prd_grp_3_coverage_pct",
    "sf_total_events",
    "sf_active_months",
    "sf_months_since_last_event",
    "sf_activity_without_recent_sales",
    "sales_without_recent_sf_activity"]

customer_context_cols = [c for c in customer_context_cols if c in df_customers.columns]

leads_long = recommendations_long.merge(
    df_customers[customer_context_cols],
    on="ID_Customer",
    how="left",
    validate="many_to_one")

display(leads_long.head())

,ID_Customer,product_group,recommended_product,model_probability,source_target_col,model_rank_per_customer,value_segment,segment_product_mean_order_value,segment_product_median_order_value,segment_product_value_events,product_mean_order_value,product_median_order_value,product_value_events,customer_mean_event_value,customer_value_events,customer_value_multiplier,value_shrinkage_weight,expected_order_value_before_customer_adjustment,expected_order_value,expected_revenue,expected_revenue_proxy,expected_revenue_is_forecast,probability_semantics,order_value_estimation_method,cohort_ML,cluster,Sales,historical_ltv,avg_monthly_revenue,active_months,months_since_last_purchase,activity_ratio_lifetime,avg_inactive_gap_months,prd_grp_1_coverage_pct,prd_grp_3_coverage_pct,sf_total_events,sf_active_months,sf_activity_without_recent_sales,sales_without_recent_sf_activity
0,C00001,grp3,Premium Standard Type 1,0.6251,purchase_flag_grp3_Premium Standard Type 1,1,Distributor,"1,495.0851","1,196.2700",137,976.8610,603.8050,546,"6,279.7074",23,2.0000,0.8204,"1,401.9910","2,803.9819","1,752.6986","1,752.6986",False,historical product-affinity score,supported segment-product estimate with shrinkage,Emerging,0,"144,433.2700","144,433.2700","4,248.0374",23,1,0.6765,0.5000,0.6667,0.1429,117,29,0,0
1,C00001,grp3,Monitoring Type 2,0.5645,purchase_flag_grp3_Monitoring Type 2,2,Distributor,"2,588.2764","1,962.5700",126,"2,215.2610","1,480.1700",1679,"6,279.7074",23,2.0000,0.8077,"2,516.5427","5,033.0853","2,841.1890","2,841.1890",False,historical product-affinity score,supported segment-product estimate with shrinkage,Emerging,0,"144,433.2700","144,433.2700","4,248.0374",23,1,0.6765,0.5000,0.6667,0.1429,117,29,0,0
2,C00001,grp1,Safety Solutions,0.5630,purchase_flag_grp1_Safety Solutions,3,Distributor,757.2329,643.1000,507,505.4395,312.6500,5642,"6,279.7074",23,2.0000,0.9441,743.1662,"1,486.3325",836.8758,836.8758,False,historical product-affinity score,supported segment-product estimate with shrinkage,Emerging,0,"144,433.2700","144,433.2700","4,248.0374",23,1,0.6765,0.5000,0.6667,0.1429,117,29,0,0
3,C00001,grp3,Installation Type 2,0.5362,purchase_flag_grp3_Installation Type 2,4,Distributor,"4,226.5728","3,277.1500",81,"3,634.1143","2,592.8300",939,"6,279.7074",23,2.0000,0.7297,"4,066.4489","8,132.8978","4,360.8969","4,360.8969",False,historical product-affinity score,supported segment-product estimate with shrinkage,Emerging,0,"144,433.2700","144,433.2700","4,248.0374",23,1,0.6765,0.5000,0.6667,0.1429,117,29,0,0
4,C00001,grp3,Training Type 2,0.5139,purchase_flag_grp3_Training Type 2,5,Distributor,"5,862.9188","4,233.7100",233,"3,552.4214","2,150.4700",1866,"6,279.7074",23,2.0000,0.8859,"5,599.3640","11,198.7280","5,755.4957","5,755.4957",False,historical product-affinity score,supported segment-product estimate with shrinkage,Emerging,0,"144,433.2700","144,433.2700","4,248.0374",23,1,0.6765,0.5000,0.6667,0.1429,117,29,0,0


## Lead Score Formula

This project uses a transparent scoring formula.
Lead Score components:
- model_score: model probability for the recommendation
- value_score: normalized customer value
- activity_score: recent and frequent purchasing behavior
- sf_engagement_score: Salesforce engagement
- risk_penalty: inactivity penalty

The weights are business assumptions. In a real company, they should be aligned with sales capacity and validated against conversion outcomes.

In [20]:
def minmax_score(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").astype(float)
    min_value = s.min(skipna=True)
    max_value = s.max(skipna=True)

    if pd.isna(min_value) or pd.isna(max_value) or max_value == min_value:
        return pd.Series(0.0, index=s.index)

    return ((s - min_value) / (max_value - min_value)).fillna(0)


leads_long["model_score"] = leads_long["model_probability"].clip(0, 1)

if "historical_ltv" in leads_long.columns:
    leads_long["value_score"] = minmax_score(leads_long["historical_ltv"])
else:
    leads_long["value_score"] = minmax_score(leads_long["Sales"])

# Recency: lower months_since_last_purchase is better.
if "months_since_last_purchase" in leads_long.columns:
    leads_long["recency_score"] = 1 - minmax_score(leads_long["months_since_last_purchase"])
else:
    leads_long["recency_score"] = 0.0

if "active_months" in leads_long.columns:
    leads_long["frequency_score"] = minmax_score(leads_long["active_months"])
else:
    leads_long["frequency_score"] = 0.0

leads_long["activity_score"] = (
    0.60 * leads_long["recency_score"]
    + 0.40 * leads_long["frequency_score"])

if "sf_total_events" in leads_long.columns:
    leads_long["sf_engagement_score"] = minmax_score(leads_long["sf_total_events"])
else:
    leads_long["sf_engagement_score"] = 0.0

if "avg_inactive_gap_months" in leads_long.columns:
    leads_long["risk_penalty"] = minmax_score(leads_long["avg_inactive_gap_months"])
else:
    leads_long["risk_penalty"] = 0.0

# transparent weighted lead score
leads_long["lead_score"] = (
    0.45 * leads_long["model_score"]
    + 0.25 * leads_long["value_score"]
    + 0.20 * leads_long["activity_score"]
    + 0.10 * leads_long["sf_engagement_score"]
    - 0.10 * leads_long["risk_penalty"]
).clip(0, 1)

display(leads_long[[
    "ID_Customer",
    "product_group",
    "recommended_product",
    "model_probability",
    "model_score",
    "value_score",
    "activity_score",
    "sf_engagement_score",
    "risk_penalty",
    "lead_score"]].head(20))

,ID_Customer,product_group,recommended_product,model_probability,model_score,value_score,activity_score,sf_engagement_score,risk_penalty,lead_score
0,C00001,grp3,Premium Standard Type 1,0.6251,0.6251,0.0947,0.8343,0.7285,0.0179,0.5429
1,C00001,grp3,Monitoring Type 2,0.5645,0.5645,0.0947,0.8343,0.7285,0.0179,0.5156
2,C00001,grp1,Safety Solutions,0.5630,0.5630,0.0947,0.8343,0.7285,0.0179,0.5150
3,C00001,grp3,Installation Type 2,0.5362,0.5362,0.0947,0.8343,0.7285,0.0179,0.5029
4,C00001,grp3,Training Type 2,0.5139,0.5139,0.0947,0.8343,0.7285,0.0179,0.4929
5,C00001,grp3,Calibration Kits Type 2,0.4949,0.4949,0.0947,0.8343,0.7285,0.0179,0.4843
6,C00001,grp3,Premium Plus Type 2,0.4949,0.4949,0.0947,0.8343,0.7285,0.0179,0.4843
7,C00001,grp1,Service Contracts,0.4669,0.4669,0.0947,0.8343,0.7285,0.0179,0.4717
8,C00001,grp3,Calibration Kits Type 1,0.4659,0.4659,0.0947,0.8343,0.7285,0.0179,0.4712
9,C00001,grp3,Core Basic Type 1,0.4406,0.4406,0.0947,0.8343,0.7285,0.0179,0.4599


## Recommendation Tiers and Hot Reasons

The quantile tiers in this section describe individual customer-product recommendations:

- Hot: top 20% of recommendation rows
- Warm: next 40%
- Cold: bottom 40%

They are stored as `recommendation_tier`. They must not be interpreted as the customer-level lead tier because each customer has many recommendations.

The actual `customer_lead_tier` is calculated later, after selecting the best recommendation per customer. This keeps the final customer distribution close to 20% Hot, 40% Warm, and 40% Cold.

The `hot_reason` column explains why an individual recommendation is attractive.

In [21]:
# Recommendation-level tier across all customer-product rows
# Ranking first avoids qcut failures when many lead scores are tied
leads_long["recommendation_tier"] = pd.qcut(
    leads_long["lead_score"].rank(method="first"),
    q=[0.0, 0.40, 0.80, 1.0],
    labels=["Cold", "Warm", "Hot"])


def build_hot_reason(row: pd.Series) -> str:
    reasons = []

    if row.get("model_score", 0) >= 0.65:
        reasons.append("high model probability")
    if row.get("value_score", 0) >= 0.70:
        reasons.append("high customer value")
    if row.get("activity_score", 0) >= 0.65:
        reasons.append("recent/frequent buyer")
    if row.get("sf_engagement_score", 0) >= 0.65:
        reasons.append("strong Salesforce engagement")
    if row.get("risk_penalty", 0) <= 0.30:
        reasons.append("low inactivity risk")
    if row.get("sf_activity_without_recent_sales", 0) == 1:
        reasons.append("recent sales activity but no recent sales")

    return "; ".join(reasons) if reasons else "moderate model fit"


leads_long["hot_reason"] = leads_long.apply(build_hot_reason, axis=1)

display(leads_long["recommendation_tier"].value_counts(dropna=False).reset_index())
display(leads_long[[
    "ID_Customer",
    "cohort_ML",
    "product_group",
    "recommended_product",
    "lead_score",
    "recommendation_tier",
    "hot_reason",
]].sort_values("lead_score", ascending=False).head(20))

,recommendation_tier,count
0,Cold,24396
1,Warm,24396
2,Hot,12198


,ID_Customer,cohort_ML,product_group,recommended_product,lead_score,recommendation_tier,hot_reason
46435,C01524,Power,grp3,Sensors Type 1,0.9116,Hot,high model probability; high customer value; r...
39236,C01288,Power,grp3,Training Type 1,0.8748,Hot,high model probability; high customer value; r...
39237,C01288,Power,grp3,Extended Support Type 2,0.8566,Hot,high model probability; high customer value; r...
8188,C00269,Power,grp3,Installation Type 1,0.8456,Hot,high model probability; recent/frequent buyer;...
38890,C01277,Power,grp3,Installation Type 1,0.8455,Hot,high model probability; recent/frequent buyer;...
47595,C01562,Power,grp1,Premium Products,0.8425,Hot,high model probability; high customer value; r...
38891,C01277,Power,grp3,Training Type 1,0.8361,Hot,high model probability; recent/frequent buyer;...
8189,C00269,Power,grp1,Service Contracts,0.8233,Hot,high model probability; recent/frequent buyer;...
1263,C00043,Power,grp3,Premium Plus Type 2,0.8182,Hot,high model probability; high customer value; r...
4276,C00141,Power,grp3,Monitoring Type 2,0.8158,Hot,high model probability; high customer value; r...


## Select final top recommendations and calculate customer tiers

A customer may have many possible recommendations. We first keep the Top-K recommendations by `lead_score`.

The customer-level tier is then calculated only from each customer's Top-1 score:

- `recommendation_tier`: strength of an individual customer-product recommendation
- `customer_lead_tier`: priority of the customer based on their best recommendation

This prevents the Top-K selection from turning every customer with at least one strong product recommendation into a disproportionately large Hot segment.

In [22]:
leads_long["revenue_rank_per_customer"] = (
    leads_long
    .groupby("ID_Customer")["expected_revenue"]
    .rank(method="first", ascending=False))

lead_generation_top_recommendations = (
    leads_long
    .sort_values(
        ["ID_Customer", "lead_score", "expected_revenue", "model_probability"],
        ascending=[True, False, False, False])
    .copy())

lead_generation_top_recommendations["lead_rank_per_customer"] = (
    lead_generation_top_recommendations
    .groupby("ID_Customer")
    .cumcount()
    + 1)

lead_generation_top_recommendations = lead_generation_top_recommendations[
    lead_generation_top_recommendations["lead_rank_per_customer"] <= TOP_K
].copy()

# One row per customer
customer_top_lead = (
    lead_generation_top_recommendations
    .loc[
        lead_generation_top_recommendations["lead_rank_per_customer"].eq(1),
        [
            "ID_Customer",
            "product_group",
            "recommended_product",
            "model_probability",
            "expected_order_value",
            "expected_revenue",
            "lead_score"]]
    .rename(columns={
        "product_group": "customer_top_product_group",
        "recommended_product": "customer_top_product",
        "model_probability": "customer_top_model_probability",
        "expected_order_value": "customer_top_expected_order_value",
        "expected_revenue": "customer_top_expected_revenue",
        "lead_score": "customer_top_lead_score"})
    .copy())

if customer_top_lead["ID_Customer"].duplicated().any():
    raise ValueError("customer_top_lead must contain exactly one row per customer.")

# Rank before qcut so ties cannot collapse quantile boundaries
customer_top_lead["customer_lead_tier"] = pd.qcut(
    customer_top_lead["customer_top_lead_score"].rank(method="first"),
    q=[0.0, 0.40, 0.80, 1.0],
    labels=["Cold", "Warm", "Hot"])

# attach the customer tier to every retained Top-K row for long-format reporting
lead_generation_top_recommendations = lead_generation_top_recommendations.merge(
    customer_top_lead[[
        "ID_Customer",
        "customer_top_lead_score",
        "customer_lead_tier"]],
    on="ID_Customer",
    how="left",
    validate="many_to_one")

display(customer_top_lead["customer_lead_tier"].value_counts(dropna=False).reset_index())
display(lead_generation_top_recommendations[[
    "ID_Customer",
    "recommended_product",
    "model_probability",
    "expected_order_value",
    "expected_revenue",
    "lead_score",
    "recommendation_tier",
    "customer_lead_tier",
    "customer_top_lead_score",
    "lead_rank_per_customer",
    "revenue_rank_per_customer"]].head(10))

,customer_lead_tier,count
0,Cold,797
1,Warm,797
2,Hot,399


,ID_Customer,recommended_product,model_probability,expected_order_value,expected_revenue,lead_score,recommendation_tier,customer_lead_tier,customer_top_lead_score,lead_rank_per_customer,revenue_rank_per_customer
0,C00001,Premium Standard Type 1,0.6251,"2,803.9819","1,752.6986",0.5429,Hot,Warm,0.5429,1,13.0000
1,C00001,Monitoring Type 2,0.5645,"5,033.0853","2,841.1890",0.5156,Hot,Warm,0.5429,2,5.0000
2,C00001,Safety Solutions,0.5630,"1,486.3325",836.8758,0.5150,Hot,Warm,0.5429,3,20.0000
3,C00002,Core Advanced Type 1,0.7298,180.9113,132.0343,0.4752,Hot,Warm,0.4752,1,20.0000
4,C00002,Calibration Kits Type 1,0.6748,205.0932,138.3905,0.4504,Hot,Warm,0.4752,2,17.0000
5,C00002,Maintenance Kits,0.6422,259.2784,166.4986,0.4357,Warm,Warm,0.4752,3,15.0000
6,C00003,Premium Custom Type 1,0.7909,"2,315.4905","1,831.2782",0.6507,Hot,Hot,0.6507,1,4.0000
7,C00003,Extended Support Type 2,0.7898,"3,883.1339","3,066.9553",0.6502,Hot,Hot,0.6507,2,1.0000
8,C00003,Sensors Type 1,0.7467,"2,573.0797","1,921.2178",0.6308,Hot,Hot,0.6507,3,3.0000
9,C00004,Premium Custom Type 1,0.5628,"4,066.6452","2,288.5251",0.5119,Hot,Warm,0.5119,1,1.0000


## Create Final Customer-Level Output

The final output contains one row per customer and combines:

- customer and cohort context
- Top-3 model recommendations by product group
- Top-3 final lead recommendations
- recommendation-level tiers for each selected product
- one customer-level priority tier based on the Top-1 lead score

In [23]:
def make_lead_topk_wide(top_recs: pd.DataFrame, top_k: int = TOP_K) -> pd.DataFrame:
    wide_parts = []

    for k in range(1, top_k + 1):
        selected_cols = [
            "ID_Customer",
            "product_group",
            "recommended_product",
            "model_probability",
            "expected_order_value",
            "expected_revenue",
            "order_value_estimation_method",
            "lead_score",
            "recommendation_tier",
            "hot_reason"]
        selected_cols = [c for c in selected_cols if c in top_recs.columns]

        kth = (
            top_recs[top_recs["lead_rank_per_customer"] == k][selected_cols]
            .rename(columns={
                "product_group": f"lead_top{k}_product_group",
                "recommended_product": f"lead_top{k}_product",
                "model_probability": f"lead_top{k}_model_probability",
                "expected_order_value": f"lead_top{k}_expected_order_value",
                "expected_revenue": f"lead_top{k}_expected_revenue",
                "order_value_estimation_method": f"lead_top{k}_value_method",
                "lead_score": f"lead_top{k}_score",
                "recommendation_tier": f"lead_top{k}_recommendation_tier",
                "hot_reason": f"lead_top{k}_reason"}))
        wide_parts.append(kth)

    final_wide = wide_parts[0]
    for kth in wide_parts[1:]:
        final_wide = final_wide.merge(
            kth,
            on="ID_Customer",
            how="outer",
            validate="one_to_one")

    return final_wide


lead_recos_wide = make_lead_topk_wide(lead_generation_top_recommendations, TOP_K)

lead_recos_wide = lead_recos_wide.merge(
    customer_top_lead[[
        "ID_Customer",
        "customer_top_lead_score",
        "customer_lead_tier"]],
    on="ID_Customer",
    how="left",
    validate="one_to_one")

# Backward-compatible Power BI alias. It now has customer-level semantics.
lead_recos_wide["lead_top1_tier"] = lead_recos_wide["customer_lead_tier"]

customer_context_final_cols = [
    c for c in [
        "ID_Customer",
        "cohort_ML",
        "cluster",
        "Sales",
        "historical_ltv",
        "avg_monthly_revenue",
        "active_months",
        "months_since_last_purchase",
        "activity_ratio_lifetime",
        "sf_total_events",
        "sf_active_months",
        "sf_months_since_last_event"]
    if c in df_customers.columns]

customer_value_export_cols = [
    "ID_Customer",
    "value_segment",
    "customer_mean_event_value",
    "customer_value_events",
    "customer_value_multiplier"]

customer_context_final = (
    df_customers[customer_context_final_cols]
    .merge(
        customer_value_profiles[customer_value_export_cols],
        on="ID_Customer",
        how="left",
        validate="one_to_one"))

lead_generation_final = (
    customer_context_final
    .merge(
        recos_grp1_top3_wide,
        on="ID_Customer",
        how="left",
        validate="one_to_one")
    .merge(
        recos_grp3_top3_wide,
        on="ID_Customer",
        how="left",
        validate="one_to_one")
    .merge(
        lead_recos_wide,
        on="ID_Customer",
        how="left",
        validate="one_to_one"))

display(lead_generation_final.head())
print("Final customer-level output:", lead_generation_final.shape)

,ID_Customer,cohort_ML,cluster,Sales,historical_ltv,avg_monthly_revenue,active_months,months_since_last_purchase,activity_ratio_lifetime,sf_total_events,sf_active_months,value_segment,customer_mean_event_value,customer_value_events,customer_value_multiplier,reco_top1_grp1_product,reco_top1_grp1_probability,reco_top1_grp1_expected_order_value,reco_top1_grp1_expected_revenue,reco_top1_grp1_value_method,reco_top1_grp1_rank,reco_top2_grp1_product,reco_top2_grp1_probability,reco_top2_grp1_expected_order_value,reco_top2_grp1_expected_revenue,reco_top2_grp1_value_method,reco_top2_grp1_rank,reco_top3_grp1_product,reco_top3_grp1_probability,reco_top3_grp1_expected_order_value,reco_top3_grp1_expected_revenue,reco_top3_grp1_value_method,reco_top3_grp1_rank,reco_top1_grp3_product,reco_top1_grp3_probability,reco_top1_grp3_expected_order_value,reco_top1_grp3_expected_revenue,reco_top1_grp3_value_method,reco_top1_grp3_rank,reco_top2_grp3_product,reco_top2_grp3_probability,reco_top2_grp3_expected_order_value,reco_top2_grp3_expected_revenue,reco_top2_grp3_value_method,reco_top2_grp3_rank,reco_top3_grp3_product,reco_top3_grp3_probability,reco_top3_grp3_expected_order_value,reco_top3_grp3_expected_revenue,reco_top3_grp3_value_method,reco_top3_grp3_rank,lead_top1_product_group,lead_top1_product,lead_top1_model_probability,lead_top1_expected_order_value,lead_top1_expected_revenue,lead_top1_value_method,lead_top1_score,lead_top1_recommendation_tier,lead_top1_reason,lead_top2_product_group,lead_top2_product,lead_top2_model_probability,lead_top2_expected_order_value,lead_top2_expected_revenue,lead_top2_value_method,lead_top2_score,lead_top2_recommendation_tier,lead_top2_reason,lead_top3_product_group,lead_top3_product,lead_top3_model_probability,lead_top3_expected_order_value,lead_top3_expected_revenue,lead_top3_value_method,lead_top3_score,lead_top3_recommendation_tier,lead_top3_reason,customer_top_lead_score,customer_lead_tier,lead_top1_tier
0,C00001,Emerging,0,"144,433.2700","144,433.2700","4,248.0374",23,1,0.6765,117,29,Distributor,"6,279.7074",23,2.0000,Safety Solutions,0.5630,"1,486.3325",836.8758,supported segment-product estimate with shrinkage,1.0000,Service Contracts,0.4669,"12,633.2832","5,897.8985",supported segment-product estimate with shrinkage,2.0000,NaN,NaN,NaN,NaN,NaN,NaN,Premium Standard Type 1,0.6251,"2,803.9819","1,752.6986",supported segment-product estimate with shrinkage,1,Monitoring Type 2,0.5645,"5,033.0853","2,841.1890",supported segment-product estimate with shrinkage,2,Installation Type 2,0.5362,"8,132.8978","4,360.8969",supported segment-product estimate with shrinkage,3,grp3,Premium Standard Type 1,0.6251,"2,803.9819","1,752.6986",supported segment-product estimate with shrinkage,0.5429,Hot,recent/frequent buyer; strong Salesforce engag...,grp3,Monitoring Type 2,0.5645,"5,033.0853","2,841.1890",supported segment-product estimate with shrinkage,0.5156,Hot,recent/frequent buyer; strong Salesforce engag...,grp1,Safety Solutions,0.5630,"1,486.3325",836.8758,supported segment-product estimate with shrinkage,0.5150,Hot,recent/frequent buyer; strong Salesforce engag...,0.5429,Warm,Warm
1,C00002,Emerging,0,"4,738.6100","4,738.6100",152.8584,8,1,0.2581,45,13,SMB,592.3262,8,0.6442,Maintenance Kits,0.6422,259.2784,166.4986,supported segment-product estimate with shrinkage,1.0000,Service Contracts,0.3371,"1,319.7295",444.8734,supported segment-product estimate with shrinkage,2.0000,NaN,NaN,NaN,NaN,NaN,NaN,Core Advanced Type 1,0.7298,180.9113,132.0343,supported segment-product estimate with shrinkage,1,Calibration Kits Type 1,0.6748,205.0932,138.3905,supported segment-product estimate with shrinkage,2,Sensors Type 2,0.6217,647.0804,402.2769,supported segment-product estimate with shrinkage,3,grp3,Core Advanced Type 1,0.7298,180.9113,132.0343,supported segment-product estimate with shrinkage,0.4752,Hot,high model probability; recent/frequent buyer;...,grp3,Calibration Kits Type 1,0.6748,205.0932,138.3905,supported segment-product est

Final customer-level output: (1993, 81)


In [24]:
display(
    lead_generation_final["customer_lead_tier"]
    .value_counts(dropna=False)
    .reset_index())

assert lead_generation_final["customer_lead_tier"].equals(
    lead_generation_final["lead_top1_tier"])

print("lead_top1_tier is a backward-compatible alias of customer_lead_tier.")

,customer_lead_tier,count
0,Cold,797
1,Warm,797
2,Hot,399


lead_top1_tier is a backward-compatible alias of customer_lead_tier.


## Final Business Summary

In [25]:
""" Useful views:
- lead tier distribution
- hot leads by cohort
- top recommended products
- average lead score by cohort """

recommendation_tier_summary = (
    leads_long
    .groupby("recommendation_tier", observed=False)
    .agg(
        recommendation_rows=("ID_Customer", "size"),
        customers=("ID_Customer", "nunique"),
        avg_lead_score=("lead_score", "mean"),
        avg_model_probability=("model_probability", "mean"))
    .reset_index())

# Keep the established output name, but give it the correct customer-level meaning.
lead_tier_summary = (
    customer_top_lead
    .groupby("customer_lead_tier", observed=False)
    .agg(
        customers=("ID_Customer", "nunique"),
        avg_top_lead_score=("customer_top_lead_score", "mean"),
        avg_top_model_probability=("customer_top_model_probability", "mean"),
        avg_top_expected_revenue=("customer_top_expected_revenue", "mean"))
    .reset_index())

customer_top_lead_context = customer_top_lead.merge(
    df_customers[["ID_Customer", "cohort_ML"]],
    on="ID_Customer",
    how="left",
    validate="one_to_one")

cohort_lead_summary = (
    customer_top_lead_context
    .groupby(["cohort_ML", "customer_lead_tier"], observed=False)
    .agg(
        customers=("ID_Customer", "nunique"),
        avg_top_lead_score=("customer_top_lead_score", "mean"),
        avg_top_expected_revenue=("customer_top_expected_revenue", "mean"))
    .reset_index()
    .sort_values(["cohort_ML", "avg_top_lead_score"], ascending=[True, False]))

product_lead_summary = (
    lead_generation_top_recommendations
    .groupby(["product_group", "recommended_product"], as_index=False)
    .agg(
        recommendation_rows=("ID_Customer", "size"),
        customers=("ID_Customer", "nunique"),
        avg_lead_score=("lead_score", "mean"),
        avg_model_probability=("model_probability", "mean"))
    .sort_values("recommendation_rows", ascending=False))

display(recommendation_tier_summary)
display(lead_tier_summary)
display(cohort_lead_summary.head(30))
display(product_lead_summary.head(30))

,recommendation_tier,recommendation_rows,customers,avg_lead_score,avg_model_probability
0,Cold,24396,1639,0.2249,0.2354
1,Warm,24396,1926,0.3712,0.4275
2,Hot,12198,1397,0.5127,0.6051


,customer_lead_tier,customers,avg_top_lead_score,avg_top_model_probability,avg_top_expected_revenue
0,Cold,797,0.4003,0.6307,"1,137.7911"
1,Warm,797,0.5150,0.7619,"2,003.0908"
2,Hot,399,0.6349,0.8108,"3,786.0650"


,cohort_ML,customer_lead_tier,customers,avg_top_lead_score,avg_top_expected_revenue
2,Core,Hot,247,0.6331,"2,939.1853"
1,Core,Warm,74,0.5353,"1,504.9903"
0,Core,Cold,1,0.4447,"2,740.8525"
4,Dormant,Warm,8,0.4968,"3,339.4429"
3,Dormant,Cold,153,0.3497,"1,550.2481"
5,Dormant,Hot,0,NaN,NaN
8,Emerging,Hot,92,0.5917,"4,161.9535"
7,Emerging,Warm,690,0.5132,"2,069.8097"
6,Emerging,Cold,627,0.4118,"1,024.4635"
11,Occasional,Hot,2,0.6080,354.0403


,product_group,recommended_product,recommendation_rows,customers,avg_lead_score,avg_model_probability
17,grp3,Core Replacement Type 2,280,280,0.4768,0.7162
26,grp3,Personal Safety Type 2,273,273,0.4564,0.6859
9,grp3,Compliance Safety Type 2,262,262,0.4558,0.6947
21,grp3,Installation Type 1,256,256,0.4907,0.7239
22,grp3,Installation Type 2,255,255,0.4662,0.6819
38,grp3,Training Type 1,240,240,0.4985,0.7147
31,grp3,Premium Standard Type 1,239,239,0.4724,0.6820
16,grp3,Core Replacement Type 1,220,220,0.4846,0.6732
19,grp3,Extended Support Type 2,220,220,0.5133,0.7335
27,grp3,Premium Custom Type 1,219,219,0.5300,0.6950


## Revenue Opportunity Summary

In [26]:
revenue_opportunity_summary = (
    leads_long
    .groupby(["product_group", "recommended_product"], as_index=False)
    .agg(
        recommendation_rows=("ID_Customer", "size"),
        customers=("ID_Customer", "nunique"),
        avg_model_probability=("model_probability", "mean"),
        avg_expected_order_value=("expected_order_value", "mean"),
        total_expected_revenue=("expected_revenue", "sum"),
        avg_expected_revenue=("expected_revenue", "mean"),
        median_value_support=("segment_product_value_events", "median"))
    .sort_values("total_expected_revenue", ascending=False))

revenue_by_lead_tier = (
    customer_top_lead
    .groupby("customer_lead_tier", observed=False, as_index=False)
    .agg(
        customers=("ID_Customer", "nunique"),
        avg_expected_order_value=("customer_top_expected_order_value", "mean"),
        total_expected_revenue=("customer_top_expected_revenue", "sum"),
        avg_expected_revenue=("customer_top_expected_revenue", "mean")))

display(revenue_opportunity_summary.head(20))
display(revenue_by_lead_tier)

,product_group,recommended_product,recommendation_rows,customers,avg_model_probability,avg_expected_order_value,total_expected_revenue,avg_expected_revenue,median_value_support
39,grp3,Training Type 1,1676,1676,0.4059,"5,195.4484","4,115,500.4656","2,455.5492",301.0000
21,grp3,Installation Type 1,1618,1618,0.3472,"6,094.1267","3,916,929.8281","2,420.8466",402.0000
19,grp3,Extended Support Type 2,1863,1863,0.3917,"4,310.3620","3,610,298.8233","1,937.8952",162.0000
22,grp3,Installation Type 2,1797,1797,0.4129,"3,891.3155","3,224,835.7107","1,794.5663",208.0000
40,grp3,Training Type 2,1621,1621,0.4300,"3,629.7002","2,831,360.7454","1,746.6754",370.0000
11,grp3,Controllers Type 2,1408,1408,0.4545,"3,343.9862","2,352,114.6008","1,670.5359",627.0000
23,grp3,Monitoring Type 1,1534,1534,0.4338,"3,250.7130","2,343,012.2883","1,527.3874",573.0000
18,grp3,Extended Support Type 1,1601,1601,0.2809,"4,617.9615","2,286,864.6691","1,428.3977",373.0000
10,grp3,Controllers Type 1,1309,1309,0.3372,"4,219.8093","2,116,931.6987","1,617.2129",921.0000
37,grp3,Sensors Type 1,1324,1324,0.4383,"3,277.2214","2,101,035.4599","1,586.8848",820.0000


,customer_lead_tier,customers,avg_expected_order_value,total_expected_revenue,avg_expected_revenue
0,Cold,797,"1,773.0697","906,819.5293","1,137.7911"
1,Warm,797,"2,609.4367","1,596,463.3907","2,003.0908"
2,Hot,399,"4,604.2757","1,510,639.9488","3,786.0650"


## Feature dictionary

In [27]:
import re


def feature_definition(
    category: str,
    business_definition: str,
    calculation: str,
    role: str,
    modeling_usage: str = "",
    caution: str = "",
) -> dict:
    return {
        "category": category,
        "business_definition": business_definition,
        "calculation": calculation,
        "role": role,
        "modeling_usage": modeling_usage,
        "caution": caution,
        "documentation_status": "Exact"}


FEATURE_DEFINITIONS = {
    # Identifiers and dates
    "ID_Customer": feature_definition(
        "Identifier", "Unique customer identifier.", "Source-system customer key; no calculation.", "Key",
        "Excluded from model features; used for joins and output grain."),
    "ID_Product": feature_definition(
        "Identifier", "Unique product/SKU identifier.", "Source-system product key; no calculation.", "Key"),
    "Date": feature_definition(
        "Time", "Date attached to the sales observation.", "Parsed with pd.to_datetime(Date).", "Raw input"),
    "YearMonth": feature_definition(
        "Time", "Calendar month of the observation.", "Date converted to monthly period: Date.dt.to_period('M').", "Derived input"),
    "Year": feature_definition(
        "Time", "Calendar year of the observation.", "Year = Date.year.", "Derived input"),
    "Month": feature_definition(
        "Time", "Calendar month number from 1 to 12.", "Month = Date.month.", "Derived input"),
    "Customer_Start_Date": feature_definition(
        "Customer attribute", "Date on which the customer relationship started.", "Source customer master-data field.", "Raw input"),

    # Raw customer and product attributes
    "Customer_Name": feature_definition(
        "Customer attribute", "Customer display name.", "Source customer master-data field.", "Raw input",
        "Not used as a model feature."),
    "Customer_Segment": feature_definition(
        "Customer attribute", "Commercial customer segment such as SMB, Mid-Market, Enterprise, or Distributor.",
        "Source customer master-data field; latest non-null value is used for value_segment.", "Raw input"),
    "Customer_Size": feature_definition(
        "Customer attribute", "Categorical customer-size classification.", "Source customer master-data field.", "Raw input"),
    "Industry": feature_definition(
        "Customer attribute", "Customer industry classification.", "Source customer master-data field.", "Raw input"),
    "Region_CC": feature_definition(
        "Geography", "Commercial-region classification.", "Source customer master-data field.", "Raw input"),
    "Region_OC": feature_definition(
        "Geography", "Operating-country or organizational-region classification.", "Source customer master-data field.", "Raw input"),
    "Country": feature_definition(
        "Geography", "Customer country.", "Source customer master-data field.", "Raw input"),
    "Acquisition_Channel": feature_definition(
        "Customer attribute", "Channel through which the customer was acquired.", "Source customer master-data field.", "Raw input"),
    "Product_Name": feature_definition(
        "Product attribute", "Product/SKU display name.", "Source product master-data field.", "Raw input"),
    "Prod_Line": feature_definition(
        "Product hierarchy", "Highest product-line classification available in the sales table.", "Source product master-data field.", "Raw input"),
    "Prd_Grp_1": feature_definition(
        "Product hierarchy", "Broad product-group level used for category recommendations.", "Source product master-data field.", "Raw input"),
    "Prd_Grp_2": feature_definition(
        "Product hierarchy", "Intermediate product-group level.", "Source product master-data field.", "Raw input"),
    "Prd_Grp_3": feature_definition(
        "Product hierarchy", "Granular product-group level used for detailed recommendations.", "Source product master-data field.", "Raw input"),
    "Unit_Price": feature_definition(
        "Sales value", "Reference or realized unit price stored on the sales row.", "Source sales field.", "Raw input"),
    "Margin_Category": feature_definition(
        "Profitability", "Categorical margin classification for the product.", "Source product/sales field.", "Raw input"),

    # Historical customer sales and breadth
    "Sales": feature_definition(
        "Customer value", "Historical sales value.",
        "In df_sales_full: recorded sales value on the row. At customer grain: SUM(Sales) across all historical rows.",
        "Model input", "Used as customer-value history, never as future revenue."),
    "Units": feature_definition(
        "Volume", "Historical quantity purchased.",
        "In df_sales_full: units on the row. At customer grain: SUM(Units) across all historical rows.",
        "Model input"),
    "order_lines": feature_definition(
        "Frequency", "Number of historical product rows for the customer.",
        "COUNT(ID_Product) by ID_Customer.", "Engineered feature",
        caution="This is an order-line proxy, not distinct orders, because no order ID is available."),
    "active_purchase_months": feature_definition(
        "Frequency", "Number of distinct months containing a purchase.",
        "NUNIQUE(YearMonth) by ID_Customer.", "Engineered feature"),
    "distinct_products": feature_definition(
        "Product breadth", "Number of distinct SKUs purchased by the customer.",
        "NUNIQUE(ID_Product) by ID_Customer.", "Engineered feature"),
    "prd_line_coverage_pct": feature_definition(
        "Product breadth", "Share of all product lines purchased by the customer.",
        "NUNIQUE(customer Prod_Line) / NUNIQUE(all Prod_Line).", "Engineered feature"),
    "prd_grp_1_coverage_pct": feature_definition(
        "Product breadth", "Share of all level-1 product groups purchased by the customer.",
        "NUNIQUE(customer Prd_Grp_1) / NUNIQUE(all Prd_Grp_1).", "Engineered feature"),
    "prd_grp_2_coverage_pct": feature_definition(
        "Product breadth", "Share of all level-2 product groups purchased by the customer.",
        "NUNIQUE(customer Prd_Grp_2) / NUNIQUE(all Prd_Grp_2).", "Engineered feature"),
    "prd_grp_3_coverage_pct": feature_definition(
        "Product breadth", "Share of all level-3 product groups purchased by the customer.",
        "NUNIQUE(customer Prd_Grp_3) / NUNIQUE(all Prd_Grp_3).", "Engineered feature"),
    "avg_order_line_value": feature_definition(
        "Customer value", "Average historical sales value per product row.",
        "Sales / order_lines; zero if the denominator is zero.", "Engineered feature",
        caution="Not a true average order value because no order ID is available."),
    "avg_unit_price_realized": feature_definition(
        "Pricing", "Average realized revenue per purchased unit.",
        "Sales / Units; zero if the denominator is zero.", "Engineered feature"),

    # Purchase activity and RFM
    "active_months": feature_definition(
        "Activity", "Number of distinct historical purchase months.",
        "NUNIQUE(YearMonth) by ID_Customer.", "Engineered feature"),
    "lifetime_months": feature_definition(
        "Tenure", "Inclusive number of months from first to last purchase.",
        "last_purchase_month_ordinal - first_purchase_month_ordinal + 1, minimum 1.", "Engineered feature"),
    "months_since_last_purchase": feature_definition(
        "Recency", "Months between the dataset's latest sales month and the customer's last purchase month.",
        "global_max_sales_month_ordinal - customer_last_month_ordinal, minimum 0.", "Engineered feature"),
    "activity_ratio_lifetime": feature_definition(
        "Activity", "Share of customer-lifetime months containing a purchase.",
        "active_months / lifetime_months, clipped to [0, 1].", "Engineered feature"),
    "max_consec_active_months": feature_definition(
        "Activity", "Longest historical streak of consecutive purchase months.",
        "Maximum run length where adjacent active month ordinals differ by 1.", "Engineered feature"),
    "current_consec_active_months": feature_definition(
        "Activity", "Consecutive active-month streak ending at the customer's most recent purchase month.",
        "Run length from the last active month backwards while adjacent month ordinals differ by 1.", "Engineered feature"),
    "avg_inactive_gap_months": feature_definition(
        "Inactivity", "Average number of inactive months between consecutive active months.",
        "MEAN(MAX(diff(active_month_ordinals) - 1, 0)).", "Engineered feature"),
    "max_inactive_gap_months": feature_definition(
        "Inactivity", "Largest number of inactive months between consecutive active months.",
        "MAX(MAX(diff(active_month_ordinals) - 1, 0)).", "Engineered feature"),
    "is_inactive_6m": feature_definition(
        "Inactivity", "Flag indicating at least six months since the last purchase.",
        "1 if months_since_last_purchase >= 6, otherwise 0.", "Engineered feature"),
    "is_inactive_12m": feature_definition(
        "Inactivity", "Flag indicating at least twelve months since the last purchase.",
        "1 if months_since_last_purchase >= 12, otherwise 0.", "Engineered feature"),
    "avg_monthly_revenue": feature_definition(
        "Customer value", "Average historical revenue per lifetime month.",
        "Sales / lifetime_months; zero if denominator is zero.", "Engineered feature"),
    "historical_ltv": feature_definition(
        "Customer value", "Historical lifetime-value proxy equal to cumulative sales.",
        "historical_ltv = Sales.", "Engineered feature",
        caution="Historical value only; it is not predictive customer lifetime value."),
    "purchase_freq_per_year": feature_definition(
        "Frequency", "Annualized frequency of active purchase months.",
        "active_months / (lifetime_months / 12).", "Engineered feature"),
    "R_recency_m": feature_definition(
        "RFM", "RFM recency component in months.", "R_recency_m = months_since_last_purchase.", "Engineered feature"),
    "F_frequency_m": feature_definition(
        "RFM", "RFM frequency component measured in active months.", "F_frequency_m = active_months.", "Engineered feature"),
    "M_sales": feature_definition(
        "RFM", "RFM monetary component measured as cumulative sales.", "M_sales = Sales.", "Engineered feature"),
    "sales_value_segment": feature_definition(
        "Segmentation", "Customer quartile based on cumulative historical sales.",
        "pd.qcut(Sales, 4) with labels Low Value, Mid Value, High Value, Top Value.", "Engineered feature"),
    "recency_segment": feature_definition(
        "Segmentation", "Categorical recency band.",
        "pd.cut(months_since_last_purchase, bins [-1,1,3,6,99]) with Very Recent, Recent, Cooling, Inactive.",
        "Engineered feature"),

    # Salesforce activity
    "sf_active_months": feature_definition(
        "Salesforce activity", "Number of distinct months containing Salesforce activity.",
        "NUNIQUE(Salesforce YearMonth) by ID_Customer.", "Engineered feature"),
    "sf_active_in_last_3m": feature_definition(
        "Salesforce activity", "Flag for recent Salesforce activity using the notebook's three-month cutoff.",
        "1 if any SF month ordinal >= global_max_SF_month_ordinal - 3, otherwise 0.", "Engineered feature",
        caution="The implementation includes the maximum month plus the prior three ordinal months."),
    "sf_active_in_last_6m": feature_definition(
        "Salesforce activity", "Flag for Salesforce activity near the latest six-month cutoff.",
        "1 if any SF month ordinal >= global_max_SF_month_ordinal - 6, otherwise 0.", "Engineered feature"),
    "sf_active_in_last_12m": feature_definition(
        "Salesforce activity", "Flag for Salesforce activity near the latest twelve-month cutoff.",
        "1 if any SF month ordinal >= global_max_SF_month_ordinal - 12, otherwise 0.", "Engineered feature"),
    "sf_total_events": feature_definition(
        "Salesforce activity", "Total Salesforce activity-event count.",
        "SUM(SF_Activity_Count) by ID_Customer.", "Engineered feature"),
    "sf_total_selling_time": feature_definition(
        "Salesforce activity", "Total recorded Salesforce selling time.",
        "SUM(SF_Selling_Time) by ID_Customer.", "Engineered feature"),
    "sf_total_activity_time": feature_definition(
        "Salesforce activity", "Total recorded Salesforce activity time.",
        "SUM(SF_Activity_Time_) by ID_Customer.", "Engineered feature"),
    "sf_activity_ratio_lifetime": feature_definition(
        "Salesforce activity", "Share of Salesforce-lifetime months with activity.",
        "sf_active_months / sf_lifetime_months, clipped to [0, 1].", "Engineered feature"),
    "sf_events_per_active_month": feature_definition(
        "Salesforce activity", "Average Salesforce events per active Salesforce month.",
        "sf_total_events / sf_active_months; zero if denominator is zero.", "Engineered feature"),
    "has_salesforce_activity": feature_definition(
        "Salesforce activity", "Flag indicating any recorded Salesforce activity.",
        "1 if sf_total_events > 0, otherwise 0.", "Engineered feature"),
    "sales_without_recent_sf_activity": feature_definition(
        "Sales activation", "Customer purchased recently but had no recent Salesforce activity.",
        "1 if active_in_last_6m = 1 and sf_active_in_last_6m = 0, otherwise 0.", "Engineered feature"),
    "sf_activity_without_recent_sales": feature_definition(
        "Sales activation", "Customer had recent Salesforce activity but no recent purchase.",
        "1 if sf_active_in_last_6m = 1 and active_in_last_6m = 0, otherwise 0.", "Engineered feature"),

    # Clustering
    "cluster": feature_definition(
        "Cohort", "Numeric cluster assigned to the customer.",
        "AgglomerativeClustering with Ward linkage and FINAL_K = 5 on standardized/encoded customer features.",
        "Model output", "Used as customer context in recommendation and reporting."),
    "cohort_ML": feature_definition(
        "Cohort", "Business-readable cohort label mapped from the numeric cluster.",
        "Rule-based label from cluster-level sales, activity, recency, product breadth, and Salesforce quantiles.",
        "Model output"),

    # Model target and registry
    "target_col": feature_definition(
        "Model metadata", "Name of the binary product-purchase target column.", "Stored target-column name.", "Diagnostic"),
    "product": feature_definition(
        "Product", "Product group represented by a target or model-registry row.",
        "Target name after removing purchase_flag hierarchy prefix.", "Model metadata"),
    "status": feature_definition(
        "Model metadata", "Training status of the product model.",
        "'trained' or 'skipped_low_support' based on class count and MIN_POSITIVES.", "Diagnostic"),
    "model": feature_definition(
        "Model metadata", "Selected classifier for the product target.",
        "Candidate with maximum validation average precision; ROC-AUC breaks ties.", "Diagnostic"),
    "n_total": feature_definition(
        "Model support", "Total customer rows available for the product model.", "COUNT(y).", "Diagnostic"),
    "positive_customers": feature_definition(
        "Model support", "Customers with a positive historical purchase target.", "SUM(binary purchase target).", "Diagnostic"),
    "negative_customers": feature_definition(
        "Model support", "Customers without the historical product purchase.", "customers - positive_customers.", "Diagnostic"),
    "positive_rate": feature_definition(
        "Model support", "Historical prevalence of the positive product target.", "MEAN(binary purchase target).", "Diagnostic"),
    "train_rows": feature_definition(
        "Model support", "Rows in the model-development training split.", "COUNT(X_train).", "Diagnostic"),
    "valid_rows": feature_definition(
        "Model support", "Rows in the validation split.", "COUNT(X_valid).", "Diagnostic"),
    "valid_positive_customers": feature_definition(
        "Model support", "Positive-target customers in the validation split.", "SUM(y_valid).", "Diagnostic"),
    "valid_avg_precision": feature_definition(
        "Model performance", "Validation average precision, summarizing the precision-recall ranking curve.",
        "sklearn.metrics.average_precision_score(y_valid, predicted_probability).", "Diagnostic"),
    "valid_roc_auc": feature_definition(
        "Model performance", "Validation ROC-AUC.",
        "sklearn.metrics.roc_auc_score(y_valid, predicted_probability).", "Diagnostic"),
    "valid_precision_at_050": feature_definition(
        "Model performance", "Validation precision at a 0.50 probability threshold.",
        "TP / (TP + FP) for predicted_probability >= 0.50.", "Diagnostic"),
    "valid_recall_at_050": feature_definition(
        "Model performance", "Validation recall at a 0.50 probability threshold.",
        "TP / (TP + FN) for predicted_probability >= 0.50.", "Diagnostic"),
    "n_numeric_features": feature_definition(
        "Model metadata", "Number of numeric input columns supplied to preprocessing.", "COUNT(numeric feature columns).", "Diagnostic"),
    "n_categorical_features": feature_definition(
        "Model metadata", "Number of categorical input columns supplied to preprocessing.", "COUNT(categorical feature columns).", "Diagnostic"),
    "baseline_avg_precision": feature_definition(
        "Model baseline", "Average precision of a constant popularity-probability baseline.",
        "Average precision using the training positive rate for every validation customer.", "Diagnostic"),
    "baseline_roc_auc": feature_definition(
        "Model baseline", "ROC-AUC of the constant popularity baseline.",
        "ROC-AUC using the training positive rate for every validation customer; normally 0.5.", "Diagnostic"),
    "avg_precision_lift_vs_baseline": feature_definition(
        "Model performance", "Relative average-precision lift over popularity baseline.",
        "valid_avg_precision / baseline_avg_precision.", "Diagnostic"),
    "targets": feature_definition(
        "Model support", "Number of product targets in a model-summary group.", "COUNT(target rows).", "Summary metric"),
    "avg_positive_rate": feature_definition(
        "Model support", "Mean target prevalence across products.", "MEAN(positive_rate).", "Summary metric"),
    "avg_valid_avg_precision": feature_definition(
        "Model performance", "Mean validation average precision across products.", "MEAN(valid_avg_precision).", "Summary metric"),
    "avg_baseline_avg_precision": feature_definition(
        "Model baseline", "Mean popularity-baseline average precision across products.", "MEAN(baseline_avg_precision).", "Summary metric"),
    "avg_precision_lift": feature_definition(
        "Model performance", "Mean average-precision lift across products.", "MEAN(avg_precision_lift_vs_baseline).", "Summary metric"),

    # Recommendation identifiers and probabilities
    "product_group": feature_definition(
        "Recommendation", "Hierarchy identifier of the recommendation: grp1 or grp3.",
        "Assigned from the model group being scored.", "Model output"),
    "recommended_product": feature_definition(
        "Recommendation", "Product group recommended to the customer.",
        "Target name after removing the purchase_flag prefix; only products not previously bought are retained.", "Model output"),
    "model_probability": feature_definition(
        "Recommendation", "Classifier score for customer-product affinity.",
        "Selected model's predict_proba(X_customer)[:, 1].", "Model output",
        caution="Current target is all-history purchase and class weights are used; treat this as an uncalibrated affinity score, not a literal future probability."),
    "source_target_col": feature_definition(
        "Model metadata", "Purchase-flag column that generated the recommendation.", "Stored source target-column name.", "Lineage"),
    "model_rank_per_customer": feature_definition(
        "Ranking", "Rank of recommendations per customer by model probability.",
        "Sort model_probability descending within ID_Customer; cumulative row number starting at 1.", "Model output"),

    # Expected order value
    "_value_event_id": feature_definition(
        "Expected value", "Identifier of the best available historical value event.",
        "Order/invoice ID when present; otherwise YearMonth derived from Date.", "Intermediate key"),
    "order_value": feature_definition(
        "Expected value", "Historical sales value of a customer-product value event.",
        "SUM(Sales) by ID_Customer, value_event_id, and product group.", "Intermediate feature"),
    "order_value_capped": feature_definition(
        "Expected value", "Winsorized historical event value used in expected-order-value estimates.",
        "For products with >=20 events, clip order_value to product 1st/99th percentiles; otherwise unchanged.", "Intermediate feature"),
    "value_segment": feature_definition(
        "Expected value", "Peer segment used to estimate order value.",
        "Latest Customer_Segment; fallback to cohort_ML; final fallback 'Unknown'.", "Intermediate feature"),
    "product_mean_order_value": feature_definition(
        "Expected value", "Mean capped event value for the recommended product across all customers.",
        "MEAN(order_value_capped) by product_group and recommended_product.", "Engineered feature"),
    "product_median_order_value": feature_definition(
        "Expected value", "Median capped event value for the recommended product across all customers.",
        "MEDIAN(order_value_capped) by product_group and recommended_product.", "Diagnostic"),
    "product_value_events": feature_definition(
        "Expected value support", "Historical value-event count for the recommended product.",
        "COUNT(order_value_capped) by product_group and recommended_product.", "Diagnostic"),
    "segment_product_mean_order_value": feature_definition(
        "Expected value", "Mean capped event value for a product within a customer peer segment.",
        "MEAN(order_value_capped) by value_segment, product_group, and recommended_product.", "Engineered feature"),
    "segment_product_median_order_value": feature_definition(
        "Expected value", "Median capped event value for a product within a customer peer segment.",
        "MEDIAN(order_value_capped) by value_segment, product_group, and recommended_product.", "Diagnostic"),
    "segment_product_value_events": feature_definition(
        "Expected value support", "Historical value-event count for a product within the peer segment.",
        "COUNT(order_value_capped) by value_segment, product_group, and recommended_product.", "Diagnostic"),
    "customer_event_value": feature_definition(
        "Expected value", "Total customer sales in one value event across products.",
        "SUM(Sales) by ID_Customer and value_event_id.", "Intermediate feature"),
    "customer_mean_event_value": feature_definition(
        "Expected value", "Customer's mean total historical value-event size.",
        "MEAN(customer_event_value) by ID_Customer.", "Engineered feature"),
    "customer_median_event_value": feature_definition(
        "Expected value", "Customer's median total historical value-event size.",
        "MEDIAN(customer_event_value) by ID_Customer.", "Diagnostic"),
    "customer_value_events": feature_definition(
        "Expected value support", "Number of historical value events observed for the customer.",
        "COUNT(customer_event_value) by ID_Customer.", "Diagnostic"),
    "segment_median_customer_event_value": feature_definition(
        "Expected value", "Typical customer mean event value within the peer segment.",
        "MEDIAN(customer_mean_event_value) by value_segment.", "Benchmark"),
    "customer_value_multiplier_raw": feature_definition(
        "Expected value", "Uncapped customer buying-power ratio relative to the peer segment.",
        "customer_mean_event_value / segment_median_customer_event_value.", "Engineered feature"),
    "customer_value_multiplier": feature_definition(
        "Expected value", "Capped customer buying-power adjustment applied to expected order value.",
        "customer_value_multiplier_raw filled with 1.0 and clipped to [0.50, 2.00].", "Engineered feature"),
    "value_shrinkage_weight": feature_definition(
        "Expected value", "Weight assigned to the segment-product estimate based on historical support.",
        "segment_product_value_events / (segment_product_value_events + 30).", "Engineered feature"),
    "expected_order_value_before_customer_adjustment": feature_definition(
        "Expected value", "Support-weighted product order-value estimate before customer adjustment.",
        "w * segment_product_mean_order_value + (1-w) * product_mean_order_value; global mean fallback.",
        "Engineered feature"),
    "expected_order_value": feature_definition(
        "Expected value", "Estimated sales value conditional on the recommended product being purchased.",
        "expected_order_value_before_customer_adjustment * customer_value_multiplier.", "Business output"),
    "expected_revenue": feature_definition(
        "Expected value", "Probability-weighted revenue opportunity for the customer-product recommendation.",
        "model_probability * expected_order_value.", "Business output",
        caution="Currently an opportunity proxy, not a finance-grade forecast, because model_probability is not a calibrated future probability."),
    "expected_revenue_proxy": feature_definition(
        "Expected value", "Explicit alias of expected_revenue under the current affinity-model semantics.",
        "expected_revenue_proxy = expected_revenue.", "Business output"),
    "expected_revenue_is_forecast": feature_definition(
        "Expected value", "Boolean indicating whether expected revenue has true forecast semantics.",
        "MODEL_PROBABILITY_IS_CALIBRATED_FUTURE_PROBABILITY; currently False.", "Interpretation flag"),
    "probability_semantics": feature_definition(
        "Expected value", "Text describing whether model_probability is affinity or calibrated future probability.",
        "Conditional label from MODEL_PROBABILITY_IS_CALIBRATED_FUTURE_PROBABILITY.", "Interpretation flag"),
    "order_value_estimation_method": feature_definition(
        "Expected value", "Lineage label describing support level and fallback used for expected order value.",
        "np.select based on segment-product event support and availability of product/global fallback.", "Lineage"),

    # Lead score
    "model_score": feature_definition(
        "Lead scoring", "Model component of the lead score.", "model_probability clipped to [0, 1].", "Score component"),
    "value_score": feature_definition(
        "Lead scoring", "Normalized customer historical-value component.",
        "Min-max normalization of historical_ltv; fallback to Sales.", "Score component"),
    "recency_score": feature_definition(
        "Lead scoring", "Normalized recency component where recent purchases score higher.",
        "1 - min-max normalization(months_since_last_purchase).", "Score component"),
    "frequency_score": feature_definition(
        "Lead scoring", "Normalized purchase-frequency component.",
        "Min-max normalization(active_months).", "Score component"),
    "activity_score": feature_definition(
        "Lead scoring", "Combined recency and frequency component.",
        "0.60 * recency_score + 0.40 * frequency_score.", "Score component"),
    "sf_engagement_score": feature_definition(
        "Lead scoring", "Normalized Salesforce-engagement component.",
        "Min-max normalization(sf_total_events); zero if unavailable.", "Score component"),
    "risk_penalty": feature_definition(
        "Lead scoring", "Normalized inactivity-gap penalty.",
        "Min-max normalization(avg_inactive_gap_months); zero if unavailable.", "Score component"),
    "lead_score": feature_definition(
        "Lead scoring", "Composite commercial-priority score for a customer-product recommendation.",
        "clip(0.45*model_score + 0.25*value_score + 0.20*activity_score + 0.10*sf_engagement_score - 0.10*risk_penalty, 0, 1).",
        "Business output"),
    "recommendation_tier": feature_definition(
        "Lead scoring", "Relative tier of an individual customer-product recommendation.",
        "qcut(rank(lead_score), 40% Cold / 40% Warm / 20% Hot) across recommendation rows.", "Business output",
        caution="This is not the customer-level tier."),
    "hot_reason": feature_definition(
        "Lead scoring", "Rule-based explanation of why a recommendation is commercially attractive.",
        "Concatenation of threshold-triggered reasons from model, value, activity, Salesforce engagement, and inactivity risk.",
        "Explainability output"),
    "revenue_rank_per_customer": feature_definition(
        "Ranking", "Within-customer rank by expected revenue.",
        "Rank expected_revenue descending within ID_Customer; method='first'.", "Business output"),
    "lead_rank_per_customer": feature_definition(
        "Ranking", "Within-customer rank by final lead priority.",
        "Sort by lead_score, expected_revenue, and model_probability descending; cumulative row number starting at 1.",
        "Business output"),
    "customer_top_lead_score": feature_definition(
        "Customer priority", "Lead score of the customer's best recommendation.",
        "lead_score where lead_rank_per_customer = 1.", "Business output"),
    "customer_lead_tier": feature_definition(
        "Customer priority", "Customer-level priority tier based on the best recommendation.",
        "qcut(rank(customer_top_lead_score), 40% Cold / 40% Warm / 20% Hot) across customers.", "Business output"),
    "lead_top1_tier": feature_definition(
        "Customer priority", "Backward-compatible alias for customer_lead_tier.",
        "lead_top1_tier = customer_lead_tier.", "Business output"),

    # Generic summary fields used in explicit output tables
    "recommendation_rows": feature_definition(
        "Summary", "Number of recommendation rows in the summary group.", "COUNT(recommendation rows).", "Summary metric"),
    "customers": feature_definition(
        "Summary", "Number of distinct customers in the summary group.", "NUNIQUE(ID_Customer).", "Summary metric"),
    "avg_lead_score": feature_definition(
        "Summary", "Mean lead score in the summary group.", "MEAN(lead_score).", "Summary metric"),
    "avg_model_probability": feature_definition(
        "Summary", "Mean model probability/affinity in the summary group.", "MEAN(model_probability).", "Summary metric"),
    "avg_top_lead_score": feature_definition(
        "Summary", "Mean Top-1 customer lead score in the summary group.", "MEAN(customer_top_lead_score).", "Summary metric"),
    "avg_top_model_probability": feature_definition(
        "Summary", "Mean Top-1 model probability in the summary group.", "MEAN(customer_top_model_probability).", "Summary metric"),
    "avg_top_expected_revenue": feature_definition(
        "Summary", "Mean Top-1 expected revenue proxy in the summary group.", "MEAN(customer_top_expected_revenue).", "Summary metric"),
    "avg_expected_order_value": feature_definition(
        "Summary", "Mean expected order value in the summary group.", "MEAN(expected_order_value).", "Summary metric"),
    "total_expected_revenue": feature_definition(
        "Summary", "Sum of expected revenue proxies in the summary group.", "SUM(expected_revenue).", "Summary metric",
        caution="Do not treat as a financial forecast until probabilities have future-horizon calibration."),
    "avg_expected_revenue": feature_definition(
        "Summary", "Mean expected revenue proxy in the summary group.", "MEAN(expected_revenue).", "Summary metric"),
    "median_value_support": feature_definition(
        "Summary", "Median segment-product event support in the summary group.", "MEDIAN(segment_product_value_events).", "Summary metric"),
    "metric": feature_definition(
        "Data quality", "Name of a quality-control metric.", "Static metric label.", "Diagnostic"),
    "value": feature_definition(
        "Data quality", "Observed value of a quality-control metric.", "Calculated value associated with metric.", "Diagnostic")}


def dynamic_feature_definition(feature: str) -> dict:
    """Return definitions for product-specific and generated wide fields."""
    match = re.match(r"^purchase_share_(grp1|grp3)_(.+)$", feature)
    if match:
        hierarchy, product_name = match.groups()
        return feature_definition(
            "Product history",
            f"Share of the customer's historical units belonging to {product_name} at {hierarchy} level.",
            f"SUM(Units for {product_name}) / SUM(Units across all products) by ID_Customer.",
            "Clustering feature",
            "Used in cohort clustering; explicitly excluded from recommendation classifiers.",
            "Directly derived from purchase history and would leak the purchase target if used in its classifier.")

    match = re.match(r"^purchase_flag_(grp1|grp3)_(.+)$", feature)
    if match:
        hierarchy, product_name = match.groups()
        return feature_definition(
            "Model target",
            f"Historical purchase indicator for {product_name} at {hierarchy} level.",
            f"MAX(1 if Units > 0 for {product_name}, else 0) by ID_Customer.",
            "Model target",
            "Separate binary target for a product-specific classifier.",
            "All-history target; it does not represent purchase in a future prediction horizon.")

    match = re.match(
        r"^reco_top(\d+)_(grp1|grp3)_(product|probability|expected_order_value|expected_revenue|value_method|rank)$",
        feature)
    if match:
        rank_number, hierarchy, metric_name = match.groups()
        metric_definitions = {
            "product": ("recommended product name", "recommended_product"),
            "probability": ("model affinity score", "model_probability"),
            "expected_order_value": ("expected order value", "expected_order_value"),
            "expected_revenue": ("expected revenue proxy", "expected_revenue"),
            "value_method": ("order-value estimation method", "order_value_estimation_method"),
            "rank": ("model-probability rank", "rank")}
        label, source = metric_definitions[metric_name]
        return feature_definition(
            "Wide recommendation output",
            f"{label.capitalize()} for model-ranked recommendation {rank_number} at {hierarchy} level.",
            f"Pivoted from {source} where within-customer {hierarchy} model rank = {rank_number}.",
            "Business output")

    match = re.match(
        r"^lead_top(\d+)_(product_group|product|model_probability|expected_order_value|expected_revenue|value_method|score|recommendation_tier|reason)$",
        feature)
    if match:
        rank_number, metric_name = match.groups()
        metric_definitions = {
            "product_group": ("product-hierarchy label", "product_group"),
            "product": ("recommended product", "recommended_product"),
            "model_probability": ("model affinity score", "model_probability"),
            "expected_order_value": ("expected order value", "expected_order_value"),
            "expected_revenue": ("expected revenue proxy", "expected_revenue"),
            "value_method": ("order-value estimation method", "order_value_estimation_method"),
            "score": ("lead score", "lead_score"),
            "recommendation_tier": ("recommendation-level tier", "recommendation_tier"),
            "reason": ("lead explanation", "hot_reason")}
        label, source = metric_definitions[metric_name]
        return feature_definition(
            "Wide lead output",
            f"{label.capitalize()} for final lead recommendation {rank_number}.",
            f"Pivoted from {source} where lead_rank_per_customer = {rank_number}.",
            "Business output")

    match = re.match(
        r"^customer_top_(product_group|product|model_probability|expected_order_value|expected_revenue)$",
        feature)
    if match:
        metric_name = match.group(1)
        source_map = {
            "product_group": "product_group",
            "product": "recommended_product",
            "model_probability": "model_probability",
            "expected_order_value": "expected_order_value",
            "expected_revenue": "expected_revenue"}
        return feature_definition(
            "Customer priority",
            f"{metric_name.replace('_', ' ').capitalize()} from the customer's Top-1 lead recommendation.",
            f"{source_map[metric_name]} where lead_rank_per_customer = 1.",
            "Business output")

    # Generic but explicit aggregation patterns for any newly added summary column.
    for prefix, operation in [("avg_", "MEAN"), ("total_", "SUM"), ("median_", "MEDIAN")]:
        if feature.startswith(prefix):
            base_feature = feature[len(prefix):]
            result = feature_definition(
                "Summary",
                f"{operation.title()} of {base_feature.replace('_', ' ')} within the table's grouping grain.",
                f"{operation}({base_feature}).",
                "Summary metric")
            result["documentation_status"] = "Pattern-derived"
            return result

    if feature.startswith("n_"):
        result = feature_definition(
            "Support", f"Count associated with {feature[2:].replace('_', ' ')}.",
            f"COUNT({feature[2:]}), as defined by the producing aggregation.", "Diagnostic")
        result["documentation_status"] = "Pattern-derived"
        return result

    result = feature_definition(
        "Other",
        f"Field '{feature}' produced by the source table shown in table_name.",
        "Direct field or producing-table calculation; inspect source notebook cell if this field was newly added.",
        "Other")
    result["documentation_status"] = "Generic fallback"
    return result

In [28]:
# Tables and their analytical grains included in the dictionary.
dictionary_sources = {
    "df_sales_full": (df_sales_full, "customer-date-product sales row"),
    "df_customers": (df_customers, "customer"),
    "cust_prod_matrix_grp_1": (cust_prod_matrix_grp_1, "customer"),
    "cust_prod_matrix_grp_3": (cust_prod_matrix_grp_3, "customer"),
    "model_registry_all_products": (model_registry_all, "product target/model"),
    "recommendations_long": (recommendations_long, "customer-product recommendation"),
    "recommendations_topk_model": (recommendations_topk_model, "customer-product Top-K model recommendation"),
    "value_events": (value_events, "customer-value-event-product"),
    "product_value_stats": (product_value_stats, "product hierarchy-product"),
    "segment_product_value_stats": (segment_product_value_stats, "segment-product hierarchy-product"),
    "customer_value_profiles": (customer_value_profiles, "customer"),
    "leads_long_scored": (leads_long, "customer-product recommendation"),
    "lead_generation_top_recommendations": (lead_generation_top_recommendations, "customer-product Top-K lead"),
    "customer_top_lead": (customer_top_lead, "customer"),
    "recos_grp1_top3_wide": (recos_grp1_top3_wide, "customer"),
    "recos_grp3_top3_wide": (recos_grp3_top3_wide, "customer"),
    "lead_generation_final": (lead_generation_final, "customer"),
    "recommendation_tier_summary": (recommendation_tier_summary, "recommendation tier"),
    "lead_tier_summary": (lead_tier_summary, "customer lead tier"),
    "cohort_lead_summary": (cohort_lead_summary, "cohort-customer lead tier"),
    "product_lead_summary": (product_lead_summary, "product hierarchy-product"),
    "expected_value_quality": (expected_value_quality, "quality metric"),
    "revenue_opportunity_summary": (revenue_opportunity_summary, "product hierarchy-product"),
    "revenue_by_lead_tier": (revenue_by_lead_tier, "customer lead tier")}


feature_dictionary_rows = []

for table_name, (table_df, grain) in dictionary_sources.items():
    for feature in table_df.columns:
        definition = FEATURE_DEFINITIONS.get(feature)
        if definition is None:
            definition = dynamic_feature_definition(feature)

        row = {
            "feature_id": f"{table_name}.{feature}",
            "table_name": table_name,
            "feature": feature,
            "data_type": str(table_df[feature].dtype),
            "grain": grain,
            **definition}
        feature_dictionary_rows.append(row)


feature_dictionary = (
    pd.DataFrame(feature_dictionary_rows)
    .sort_values(["table_name", "category", "feature"])
    .reset_index(drop=True))

# Coverage diagnostics: every table column must have exactly one non-empty dictionary row.
source_column_count = sum(len(table_df.columns) for table_df, _ in dictionary_sources.values())

feature_dictionary_coverage = pd.DataFrame({
    "metric": [
        "source table-column combinations",
        "dictionary rows",
        "duplicate feature IDs",
        "missing business definitions",
        "missing calculations",
        "generic fallback rows"],
    "value": [
        source_column_count,
        len(feature_dictionary),
        feature_dictionary["feature_id"].duplicated().sum(),
        feature_dictionary["business_definition"].isna().sum()
            + feature_dictionary["business_definition"].eq("").sum(),
        feature_dictionary["calculation"].isna().sum()
            + feature_dictionary["calculation"].eq("").sum(),
        feature_dictionary["documentation_status"].eq("Generic fallback").sum()]})

assert len(feature_dictionary) == source_column_count
assert feature_dictionary["feature_id"].is_unique
assert feature_dictionary["business_definition"].fillna("").ne("").all()
assert feature_dictionary["calculation"].fillna("").ne("").all()

display(feature_dictionary_coverage)
display(feature_dictionary.head(10))

generic_dictionary_rows = feature_dictionary[
    feature_dictionary["documentation_status"].eq("Generic fallback")]

if not generic_dictionary_rows.empty:
    print("Review newly added fields using generic fallback definitions:")
    display(generic_dictionary_rows[["table_name", "feature", "calculation"]])
else:
    print("All current fields have exact or pattern-derived definitions.")

,metric,value
0,source table-column combinations,513
1,dictionary rows,513
2,duplicate feature IDs,0
3,missing business definitions,0
4,missing calculations,0
5,generic fallback rows,0


,feature_id,table_name,feature,data_type,grain,category,business_definition,calculation,role,modeling_usage,caution,documentation_status
0,cohort_lead_summary.cohort_ML,cohort_lead_summary,cohort_ML,object,cohort-customer lead tier,Cohort,Business-readable cohort label mapped from the...,"Rule-based label from cluster-level sales, act...",Model output,,,Exact
1,cohort_lead_summary.customer_lead_tier,cohort_lead_summary,customer_lead_tier,category,cohort-customer lead tier,Customer priority,Customer-level priority tier based on the best...,"qcut(rank(customer_top_lead_score), 40% Cold /...",Business output,,,Exact
2,cohort_lead_summary.avg_top_expected_revenue,cohort_lead_summary,avg_top_expected_revenue,float64,cohort-customer lead tier,Summary,Mean Top-1 expected revenue proxy in the summa...,MEAN(customer_top_expected_revenue).,Summary metric,,,Exact
3,cohort_lead_summary.avg_top_lead_score,cohort_lead_summary,avg_top_lead_score,float64,cohort-customer lead tier,Summary,Mean Top-1 customer lead score in the summary ...,MEAN(customer_top_lead_score).,Summary metric,,,Exact
4,cohort_lead_summary.customers,cohort_lead_summary,customers,int64,cohort-customer lead tier,Summary,Number of distinct customers in the summary gr...,NUNIQUE(ID_Customer).,Summary metric,,,Exact
5,cust_prod_matrix_grp_1.ID_Customer,cust_prod_matrix_grp_1,ID_Customer,object,customer,Identifier,Unique customer identifier.,Source-system customer key; no calculation.,Key,Excluded from model features; used for joins a...,,Exact
6,cust_prod_matrix_grp_1.purchase_flag_grp1_Auto...,cust_prod_matrix_grp_1,purchase_flag_grp1_Automation Modules,int64,customer,Model target,Historical purchase indicator for Automation M...,"MAX(1 if Units > 0 for Automation Modules, els...",Model target,Separate binary target for a product-specific ...,All-history target; it does not represent purc...,Exact
7,cust_prod_matrix_grp_1.purchase_flag_grp1_Core...,cust_prod_matrix_grp_1,purchase_flag_grp1_Core Products,int64,customer,Model target,Historical purchase indicator for Core Product...,"MAX(1 if Units > 0 for Core Products, else 0) ...",Model target,Separate binary target for a product-specific ...,All-history target; it does not represent purc...,Exact
8,cust_prod_matrix_grp_1.purchase_flag_grp1_Main...,cust_prod_matrix_grp_1,purchase_flag_grp1_Maintenance Kits,int64,customer,Model target,Historical purchase indicator for Maintenance ...,"MAX(1 if Units > 0 for Maintenance Kits, else ...",Model target,Separate binary target for a product-specific ...,All-history target; it does not represent purc...,Exact
9,cust_prod_matrix_grp_1.purchase_flag_grp1_Prem...,cust_prod_matrix_grp_1,purchase_flag_grp1_Premium Products,int64,customer,Model target,Historical purchase indicator for Premium Prod...,"MAX(1 if Units > 0 for Premium Products, else ...",Model target,Separate binary target for a product-specific ...,All-history target; it does not represent purc...,Exact


All current fields have exact or pattern-derived definitions.


In [29]:
# save Outputs
outputs = {
    "recos_grp1_top3_wide": recos_grp1_top3_wide,
    "recos_grp3_top3_wide": recos_grp3_top3_wide,
    "recommendations_long": recommendations_long,
    "recommendations_topk_model": recommendations_topk_model,
    "model_registry_all_products": model_registry_all,
    "leads_long_scored": leads_long,
    "lead_generation_top_recommendations": lead_generation_top_recommendations,
    "lead_generation_final": lead_generation_final,
    "recommendation_tier_summary": recommendation_tier_summary,
    "lead_tier_summary": lead_tier_summary,
    "customer_top_lead": customer_top_lead,
    "cohort_lead_summary": cohort_lead_summary,
    "product_lead_summary": product_lead_summary,
    "expected_value_quality": expected_value_quality,
    "product_value_stats": product_value_stats,
    "segment_product_value_stats": segment_product_value_stats,
    "customer_value_profiles": customer_value_profiles,
    "revenue_opportunity_summary": revenue_opportunity_summary,
    "revenue_by_lead_tier": revenue_by_lead_tier,
    "feature_dictionary": feature_dictionary,
    "feature_dictionary_coverage": feature_dictionary_coverage}

for name, df in outputs.items():
    df.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
    df.to_excel(OUTPUT_DIR / f"{name}.xlsx", index=False)

print("Saved files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path.name)

Saved files:
- cohort_lead_summary.csv
- cohort_lead_summary.xlsx
- customer_top_lead.csv
- customer_top_lead.xlsx
- customer_value_profiles.csv
- customer_value_profiles.xlsx
- expected_value_quality.csv
- expected_value_quality.xlsx
- feature_dictionary.csv
- feature_dictionary.xlsx
- feature_dictionary_coverage.csv
- feature_dictionary_coverage.xlsx
- lead_generation_final.csv
- lead_generation_final.xlsx
- lead_generation_top_recommendations.csv
- lead_generation_top_recommendations.xlsx
- lead_tier_summary.csv
- lead_tier_summary.xlsx
- leads_long_scored.csv
- leads_long_scored.xlsx
- model_registry_all_products.csv
- model_registry_all_products.xlsx
- product_lead_summary.csv
- product_lead_summary.xlsx
- product_value_stats.csv
- product_value_stats.xlsx
- recommendation_tier_summary.csv
- recommendation_tier_summary.xlsx
- recommendations_long.csv
- recommendations_long.xlsx
- recommendations_topk_model.csv
- recommendations_topk_model.xlsx
- recos_grp1_top3_wide.csv
- recos_gr